# Comparação — BRICS AI Governance Declaration × OECD Recommendation on AI

Este notebook compara os dois documentos já extraídos (`BRICS.json` e
`OCDE.json`) em dois eixos:

1. **Similaridade textual** — o quanto o vocabulário e a ênfase temática dos
   dois documentos se sobrepõem, usando o índice de Jaccard (sobreposição de
   vocabulário) e a similaridade de cosseno (sobreposição ponderada por
   frequência dos termos);
2. **Divergência/convergência por termo** — quais termos cada documento
   prioriza mais (gráfico *tornado*) e como os termos compartilhados se
   distribuem entre os dois documentos (dispersão BRICS × OCDE).

## Importação, carregamento e tokenização

In [ ]:
import json
import math
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

STOPWORDS = {
    "the", "a", "an", "and", "of", "to", "in", "that", "for", "as", "on",
    "with", "by", "or", "be", "is", "are", "we", "our", "their", "all",
    "at", "from", "this", "these", "those", "such", "which", "it", "its",
    "within", "while", "also", "both", "other", "than", "then", "so",
    "but", "not", "no", "do", "does", "did", "has", "have", "had", "was",
    "were", "been", "being", "will", "would", "should", "could", "can",
    "may", "might", "must", "shall", "us", "any", "into", "across", "over",
    "under", "about", "between", "including", "particularly", "especially",
    "towards", "toward", "through", "among", "who", "what", "how", "if",
    "each", "more", "most", "some", "own", "they", "them", "there",
}

def load_tokens(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    text = " ".join(section["text"] for section in data["sections"])
    tokens = re.findall(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2], data

BRICS_LABEL = "BRICS Declaration"
OECD_LABEL = "OECD Recommendation"

brics_tokens, brics_data = load_tokens("BRICS.json")
ocde_tokens, ocde_data = load_tokens("OCDE.json")

brics_counts = Counter(brics_tokens)
ocde_counts = Counter(ocde_tokens)

print(f"{BRICS_LABEL}: {len(brics_tokens)} tokens, {len(brics_counts)} termos únicos")
print(f"{OECD_LABEL}: {len(ocde_tokens)} tokens, {len(ocde_counts)} termos únicos")


## 1. Similaridade entre os textos

Duas métricas complementares:

- **Índice de Jaccard** — proporção de termos que aparecem nos *dois*
  documentos em relação ao total de termos distintos usados por qualquer um
  deles. Mede sobreposição pura de vocabulário, sem considerar frequência.
- **Similaridade de cosseno** — trata cada documento como um vetor de
  frequências relativas (por 1.000 tokens) sobre o vocabulário combinado.
  Mede o quanto os dois documentos "apontam na mesma direção" temática,
  dando mais peso a termos usados com frequência semelhante nos dois.

In [ ]:
def relative_freq(counter, total_tokens, term):
    return counter.get(term, 0) / total_tokens * 1000

vocab_brics = set(brics_counts)
vocab_ocde = set(ocde_counts)
vocab_union = vocab_brics | vocab_ocde
vocab_intersection = vocab_brics & vocab_ocde

jaccard = len(vocab_intersection) / len(vocab_union)

terms = sorted(vocab_union)
vec_brics = [relative_freq(brics_counts, len(brics_tokens), t) for t in terms]
vec_ocde = [relative_freq(ocde_counts, len(ocde_tokens), t) for t in terms]

dot = sum(a * b for a, b in zip(vec_brics, vec_ocde))
norm_brics = math.sqrt(sum(a * a for a in vec_brics))
norm_ocde = math.sqrt(sum(b * b for b in vec_ocde))
cosine_similarity = dot / (norm_brics * norm_ocde)

print(f"Vocabulário BRICS:        {len(vocab_brics)} termos")
print(f"Vocabulário OCDE:         {len(vocab_ocde)} termos")
print(f"Termos em comum:         {len(vocab_intersection)} termos")
print(f"Índice de Jaccard:       {jaccard:.3f}  ({jaccard*100:.1f}% do vocabulário combinado é compartilhado)")
print(f"Similaridade de cosseno: {cosine_similarity:.3f}  (1 = idêntico, 0 = nenhuma sobreposição de ênfase)")


### Gráfico — as duas métricas de similaridade lado a lado

Ambas as métricas ficam entre 0 e 1, o que permite compará-las diretamente
na mesma escala.

In [ ]:
SEQUENTIAL_BLUE = "#2a78d6"
INK_PRIMARY = "#0b0b0b"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
SURFACE = "#fcfcfb"

def plot_ranked_bar(pairs, title, xlabel="Frequência (nº de ocorrências)", xlim_max=None, value_fmt="{:.0f}", filename=None):
    labels = [p[0] for p in pairs][::-1]
    values = [p[1] for p in pairs][::-1]

    fig, ax = plt.subplots(figsize=(8, max(2.2, 0.45 * len(pairs))), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    bars = ax.barh(labels, values, color=SEQUENTIAL_BLUE, height=0.55, zorder=3)

    ax.set_xlabel(xlabel, color=INK_MUTED, fontsize=10)
    ax.set_title(title, color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

    ax.tick_params(axis="y", colors=INK_PRIMARY, labelsize=10)
    ax.tick_params(axis="x", colors=INK_MUTED, labelsize=9)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)

    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    max_val = xlim_max if xlim_max is not None else max(values)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_width() + max_val * 0.015,
            bar.get_y() + bar.get_height() / 2,
            value_fmt.format(value),
            va="center", ha="left",
            color=INK_PRIMARY, fontsize=9,
        )
    ax.set_xlim(0, max_val * 1.15)

    fig.tight_layout()
    if filename:
        fig.savefig(filename, dpi=150, facecolor=SURFACE, bbox_inches="tight")
    plt.show()

plot_ranked_bar(
    [("Índice de Jaccard\n(sobreposição de vocabulário)", jaccard),
     ("Similaridade de cosseno\n(sobreposição de ênfase)", cosine_similarity)],
    "Similaridade textual — BRICS Declaration × OECD Recommendation",
    xlabel="Similaridade (0 a 1)",
    xlim_max=1.0,
    value_fmt="{:.3f}",
)


### Termos mais compartilhados

Os termos com maior frequência combinada nos dois documentos — o núcleo de
vocabulário que os dois textos têm efetivamente em comum.

In [ ]:
MIN_COMBINED_COUNT = 4

shared_terms = [
    (t, brics_counts[t] + ocde_counts[t])
    for t in vocab_intersection
    if brics_counts[t] + ocde_counts[t] >= MIN_COMBINED_COUNT
]
shared_terms.sort(key=lambda x: x[1], reverse=True)

plot_ranked_bar(
    shared_terms[:15],
    "Termos mais compartilhados entre os dois documentos (nº total de ocorrências)",
)


## 2. Divergência e convergência por termo

### Gráfico tornado — quem prioriza cada termo

Para os termos com pelo menos 4 ocorrências combinadas, comparamos a
frequência relativa (por 1.000 tokens) em cada documento. O gráfico mostra,
de um lado, os termos mais característicos do BRICS e, do outro, os mais
característicos da OCDE — ordenados pela diferença entre os dois.

In [ ]:
CATEGORICAL_BLUE = "#2a78d6"   # BRICS
CATEGORICAL_AQUA = "#1baf7a"   # OECD
TOP_N_EACH_SIDE = 12

rows = []
for t in vocab_union:
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    combined_count = brics_counts.get(t, 0) + ocde_counts.get(t, 0)
    if combined_count >= MIN_COMBINED_COUNT:
        rows.append((t, b, o, b - o))

rows.sort(key=lambda x: x[3])
oecd_leaning = rows[:TOP_N_EACH_SIDE]                 # diferença mais negativa -> OCDE prioriza mais
brics_leaning = rows[-TOP_N_EACH_SIDE:]               # diferença mais positiva -> BRICS prioriza mais
tornado_rows = oecd_leaning + brics_leaning
tornado_rows.sort(key=lambda x: x[3])                 # OCDE no topo, BRICS na base

labels = [r[0] for r in tornado_rows]
brics_vals = [r[1] for r in tornado_rows]
ocde_vals = [r[2] for r in tornado_rows]

fig, ax = plt.subplots(figsize=(9, 0.4 * len(tornado_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(tornado_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    "Termos que cada documento prioriza — BRICS (esquerda) × OCDE (direita)",
    color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals))
ax.set_xlim(-max_abs * 1.15, max_abs * 1.15)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### Dispersão BRICS × OCDE — convergência e divergência

Cada ponto é um termo (com pelo menos 4 ocorrências combinadas), posicionado
pela frequência relativa no BRICS (eixo X) e na OCDE (eixo Y). A linha
diagonal tracejada marca "mesma prioridade nos dois documentos": pontos
próximos a ela **convergem**; pontos afastados da diagonal — grudados em um
dos eixos — **divergem**, revelando o vocabulário específico de cada
documento.

In [ ]:
CATEGORICAL_YELLOW = "#eda100"  # termos convergentes (equilíbrio entre os dois documentos)
BALANCE_RATIO_THRESHOLD = 0.6    # min(b, o) / max(b, o) >= 0.6 => convergente

scatter_rows = [r for r in rows]  # já filtrado por MIN_COMBINED_COUNT acima

def classify(b, o):
    ratio = min(b, o) / max(b, o) if max(b, o) > 0 else 1.0
    if ratio >= BALANCE_RATIO_THRESHOLD:
        return "convergente"
    return "BRICS" if b > o else "OCDE"

groups = {"convergente": [], "BRICS": [], "OCDE": []}
for t, b, o, _ in scatter_rows:
    groups[classify(b, o)].append((t, b, o))

fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

max_val = max(max(b, o) for _, b, o, _ in scatter_rows) * 1.1
ax.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

color_map = {"BRICS": CATEGORICAL_BLUE, "OCDE": CATEGORICAL_AQUA, "convergente": CATEGORICAL_YELLOW}
label_map = {"BRICS": BRICS_LABEL, "OCDE": OECD_LABEL, "convergente": "Convergente (equilíbrio)"}

for group_name in ["convergente", "BRICS", "OCDE"]:
    pts = groups[group_name]
    if not pts:
        continue
    xs = [p[1] for p in pts]
    ys = [p[2] for p in pts]
    ax.scatter(xs, ys, s=46, color=color_map[group_name], alpha=0.85, zorder=3,
               edgecolors=SURFACE, linewidths=0.6, label=label_map[group_name])

# Rotula os termos mais extremos em cada grupo para dar contexto ao gráfico
def top_extreme(pts, key, n=4):
    return sorted(pts, key=key, reverse=True)[:n]

annotate_targets = (
    top_extreme(groups["BRICS"], key=lambda p: p[1] - p[2])
    + top_extreme(groups["OCDE"], key=lambda p: p[2] - p[1])
    + top_extreme(groups["convergente"], key=lambda p: p[1] + p[2])
)
for t, x, y in annotate_targets:
    ax.annotate(t, (x, y), textcoords="offset points", xytext=(6, 4),
                fontsize=8, color=INK_PRIMARY)

ax.set_xlabel(f"Frequência relativa — {BRICS_LABEL} (por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Frequência relativa — {OECD_LABEL} (por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title("Convergência e divergência de termos — BRICS × OCDE", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)

ax.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.text(max_val * 0.02, max_val * 0.97, "↑ mais enfatizado pela OCDE", color=INK_MUTED, fontsize=9, va="top")
ax.text(max_val * 0.97, max_val * 0.03, "→ mais enfatizado pelo BRICS", color=INK_MUTED, fontsize=9, ha="right")

ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=9, bbox_to_anchor=(0.02, 0.9))

fig.tight_layout()
plt.show()


## Síntese

- A similaridade de vocabulário bruta (Jaccard) é moderada-baixa e a
  similaridade de cosseno (que pondera pela frequência) é moderada — os dois
  documentos tratam do mesmo tema (governança de IA), mas com vocabulário e
  ênfases distintos.
- O **BRICS** prioriza termos ligados a desenvolvimento, países em
  desenvolvimento, economia digital e cooperação internacional
  (*countries*, *digital*, *development*, *developing*, *economy*,
  *technologies*, *innovation*, *international*, *governance*).
- A **OCDE** prioriza termos ligados à arquitetura de governança do próprio
  sistema de IA e aos atores responsáveis por ele (*system*, *lifecycle*,
  *actors*, *stakeholders*, *governments*, *trustworthy*, *policy*,
  *recommendation*, *principles*).
- Termos como *development*, *systems*, *data*, *digital*, *rights* e
  *governance* aparecem com peso relevante nos dois textos e formam o núcleo
  comum de vocabulário entre as duas declarações.

## 3. PBIA — a qual dos dois documentos o Brasil se alinha mais?

O PBIA (`PBIA.json`) é o Plano Brasileiro de Inteligência Artificial. Ele é
tokenizado com exatamente o mesmo método usado para BRICS e OCDE (mesma lista
de *stopwords*, mesmo campo `sections`), para que as três séries de
frequência sejam diretamente comparáveis. A pergunta desta seção: o
vocabulário do PBIA se aproxima mais do eixo **desenvolvimento / países em
desenvolvimento / economia digital** do BRICS, ou do eixo **arquitetura de
governança / atores responsáveis** da OCDE?

In [ ]:
PBIA_LABEL = "PBIA (Brasil)"

pbia_tokens, pbia_data = load_tokens("PBIA.json")
pbia_counts = Counter(pbia_tokens)

print(f"{PBIA_LABEL}: {len(pbia_tokens)} tokens, {len(pbia_counts)} termos únicos")


### 3.1. Similaridade do PBIA com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (PBIA, BRICS) e (PBIA, OCDE).

In [ ]:
def jaccard_cosine(counts_a, tokens_a, counts_b, tokens_b):
    vocab_a = set(counts_a)
    vocab_b = set(counts_b)
    union = vocab_a | vocab_b
    inter = vocab_a & vocab_b
    jac = len(inter) / len(union)

    terms_u = sorted(union)
    va = [relative_freq(counts_a, len(tokens_a), t) for t in terms_u]
    vb = [relative_freq(counts_b, len(tokens_b), t) for t in terms_u]
    dot = sum(x * y for x, y in zip(va, vb))
    norm_a = math.sqrt(sum(x * x for x in va))
    norm_b = math.sqrt(sum(y * y for y in vb))
    cos = dot / (norm_a * norm_b) if norm_a and norm_b else 0.0
    return jac, cos

jaccard_pbia_brics, cosine_pbia_brics = jaccard_cosine(pbia_counts, pbia_tokens, brics_counts, brics_tokens)
jaccard_pbia_ocde, cosine_pbia_ocde = jaccard_cosine(pbia_counts, pbia_tokens, ocde_counts, ocde_tokens)

print(f"PBIA × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_pbia_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_pbia_brics:.3f}")
print()
print(f"PBIA × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_pbia_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_pbia_ocde:.3f}")
print()
print(f"(referência) {BRICS_LABEL} × {OECD_LABEL}: Jaccard={jaccard:.3f}  Cosseno={cosine_similarity:.3f}")


### Gráfico — com qual documento o PBIA se parece mais?

As mesmas duas métricas, lado a lado, para os dois pares de comparação.

In [ ]:
CATEGORICAL_PBIA = "#e2574c"  # cor de identidade do PBIA nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_pbia_brics, cosine_pbia_brics]
ocde_values = [jaccard_pbia_ocde, cosine_pbia_ocde]

fig, ax = plt.subplots(figsize=(8, 5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.32

ax.barh([y + bar_h / 2 + 0.02 for y in y_pos], brics_values, height=bar_h,
        color=CATEGORICAL_BLUE, zorder=3, label=f"PBIA × {BRICS_LABEL}")
ax.barh([y - bar_h / 2 - 0.02 for y in y_pos], ocde_values, height=bar_h,
        color=CATEGORICAL_AQUA, zorder=3, label=f"PBIA × {OECD_LABEL}")

for y, v in zip(y_pos, brics_values):
    ax.text(v + 0.012, y + bar_h / 2 + 0.02, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)
for y, v in zip(y_pos, ocde_values):
    ax.text(v + 0.012, y - bar_h / 2 - 0.02, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_xlim(0, max(brics_values + ocde_values) * 1.3)
ax.set_title("Com qual documento o PBIA se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 3.2. O vocabulário mais usado pelo PBIA ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do PBIA (excluindo palavras autorreferentes
como *brazil*, *brazilian* e *pbia*), comparamos o quanto **BRICS** e
**OCDE** usam cada um desses mesmos termos. Isso revela se o vocabulário que
o Brasil prioriza no seu próprio plano é mais eco do discurso do BRICS ou da
OCDE.

In [ ]:
SELF_REFERENTIAL = {"pbia", "brazil", "brazilian"}
MIN_PBIA_COUNT = 3
TOP_N_PBIA_TERMS = 18

pbia_terms_rows = []
for t, pbia_c in pbia_counts.most_common():
    if t in SELF_REFERENTIAL or pbia_c < MIN_PBIA_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    pbia_terms_rows.append((t, b, o, b - o))
    if len(pbia_terms_rows) >= TOP_N_PBIA_TERMS:
        break

pbia_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in pbia_terms_rows]
brics_vals = [r[1] for r in pbia_terms_rows]
ocde_vals = [r[2] for r in pbia_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(pbia_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(pbia_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo PBIA: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 3.3. Mapa de alinhamento — PBIA entre BRICS e OCDE

Eixo X: similaridade do PBIA com o **BRICS Declaration**. Eixo Y:
similaridade do PBIA com a **OECD Recommendation**. O PBIA aparece como uma
bola (uma para cada métrica — cosseno, o ponto principal, e Jaccard, o ponto
menor e mais claro) posicionada pelas similaridades calculadas na Seção 3.1:
quanto mais **à direita** (eixo X), mais alinhado ao BRICS; quanto mais
**para cima** (eixo Y), mais alinhado à OCDE. Como referência, marcamos
também onde o próprio BRICS e a própria OCDE cairiam neste mapa (cada
documento tem similaridade 1 consigo mesmo e a similaridade cruzada
BRICS×OCDE calculada na Seção 1 com o outro).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (similaridade de cosseno) e ponto secundário (Jaccard), ligados por uma linha fina
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)

ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA no mapa BRICS × OCDE — a qual documento o Brasil se aproxima mais?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

ax.text(0.02, axis_max * 0.98, "↑ mais alinhado à OCDE", color=INK_MUTED, fontsize=9, va="top")
ax.text(axis_max * 0.98, 0.02, "→ mais alinhado ao BRICS", color=INK_MUTED, fontsize=9, ha="right")

ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5, bbox_to_anchor=(0.02, 0.86))

fig.tight_layout()
plt.show()


## Síntese — PBIA, BRICS e OCDE

- Nas duas métricas, o **PBIA se aproxima mais do BRICS do que da OCDE**:
  similaridade de cosseno de **0,54** com o BRICS contra **0,40** com a OCDE,
  e Jaccard de **0,22** contra **0,20**. A diferença é mais nítida na
  similaridade de cosseno (que pondera pela frequência de uso), sugerindo
  que a ênfase temática do PBIA — não apenas o vocabulário bruto — segue
  mais de perto o BRICS.
- O vocabulário mais usado pelo PBIA (*development*, *artificial*,
  *intelligence*, *technological*, *national*, *data*, *innovation*,
  *technologies*, *country*, *global*) ecoa fortemente no BRICS, que
  compartilha a mesma ênfase em desenvolvimento tecnológico nacional,
  inovação e cooperação entre países em desenvolvimento.
- Ainda assim, o PBIA carrega um traço de vocabulário mais próximo da OCDE
  em termos como *responsible*, *plan*, *economic* e *use* — reflexo direto
  do Eixo 5 do Plano ("Support for the Regulatory and Governance Process of
  AI") e dos princípios de *Responsible AI* descritos no documento, que
  dialogam com a linguagem de governança e responsabilidade típica da OCDE.
- No mapa BRICS × OCDE, a bola do PBIA fica claramente **puxada para o eixo
  X** (mais perto do BRICS do que da OCDE), mas ainda distante do canto
  onde o próprio BRICS estaria — evidência de que o PBIA tem um vocabulário
  próprio e mais amplo (ações, planos, setores, investimento), não sendo uma
  cópia do discurso de nenhum dos dois blocos, mas com uma inclinação clara
  para a agenda de desenvolvimento tecnológico nacional que caracteriza o
  BRICS.

## 4. AI+ (China) — a qual documento a política chinesa se aproxima mais?

O AI+ (`AI_PLUS.json`) é a política *Opinions of the State Council on
Deepening the Implementation of the "Artificial Intelligence+" Initiative*,
do Conselho de Estado da China. Ele é tokenizado com exatamente o mesmo
método usado para BRICS, OCDE e PBIA (mesma lista de *stopwords*, mesmo
campo `sections`, mesmo filtro de tamanho mínimo de palavra), para que todas
as séries de frequência sejam diretamente comparáveis.

Comparamos o AI+ com os três documentos já carregados — BRICS Declaration,
OECD Recommendation e PBIA — para responder: o vocabulário e a ênfase
temática do AI+ se aproximam mais do eixo **desenvolvimento / países em
desenvolvimento / economia digital** (BRICS), do eixo **arquitetura de
governança / atores responsáveis** (OCDE), ou do plano nacional brasileiro
(PBIA)?

In [ ]:
AIPLUS_LABEL = "AI+ (China)"

aiplus_tokens, aiplus_data = load_tokens("AI_PLUS.json")
aiplus_counts = Counter(aiplus_tokens)

print(f"{AIPLUS_LABEL}: {len(aiplus_tokens)} tokens, {len(aiplus_counts)} termos únicos")


### 4.1. Similaridade do AI+ com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (AI+, BRICS), (AI+, OCDE) e (AI+, PBIA).

In [ ]:
jaccard_aiplus_brics, cosine_aiplus_brics = jaccard_cosine(aiplus_counts, aiplus_tokens, brics_counts, brics_tokens)
jaccard_aiplus_ocde, cosine_aiplus_ocde = jaccard_cosine(aiplus_counts, aiplus_tokens, ocde_counts, ocde_tokens)
jaccard_aiplus_pbia, cosine_aiplus_pbia = jaccard_cosine(aiplus_counts, aiplus_tokens, pbia_counts, pbia_tokens)

print(f"{AIPLUS_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_aiplus_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_aiplus_brics:.3f}")
print()
print(f"{AIPLUS_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_aiplus_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_aiplus_ocde:.3f}")
print()
print(f"{AIPLUS_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_aiplus_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_aiplus_pbia:.3f}")


### Gráfico — com qual documento o AI+ se parece mais?

As três comparações lado a lado (AI+ × BRICS, AI+ × OCDE, AI+ × PBIA), para
as duas métricas.

In [ ]:
CATEGORICAL_AIPLUS = "#8b5cf6"  # cor de identidade do AI+ nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_aiplus_brics, cosine_aiplus_brics]
ocde_values = [jaccard_aiplus_ocde, cosine_aiplus_ocde]
pbia_values = [jaccard_aiplus_pbia, cosine_aiplus_pbia]

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.24
offsets = [bar_h + 0.03, 0.0, -(bar_h + 0.03)]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{AIPLUS_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{AIPLUS_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{AIPLUS_LABEL} × {PBIA_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values
ax.set_xlim(0, max(all_values) * 1.35)
ax.set_title("Com qual documento o AI+ se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 4.2. O vocabulário mais usado pelo AI+ ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do AI+ (excluindo palavras autorreferentes
como *china* e *chinese*), comparamos o quanto **BRICS** e **OCDE** usam
cada um desses mesmos termos — a mesma análise feita para o PBIA na Seção
3.2, agora para a política chinesa.

In [ ]:
SELF_REFERENTIAL_AIPLUS = {"china", "chinese"}
MIN_AIPLUS_COUNT = 3
TOP_N_AIPLUS_TERMS = 18

aiplus_terms_rows = []
for t, aiplus_c in aiplus_counts.most_common():
    if t in SELF_REFERENTIAL_AIPLUS or aiplus_c < MIN_AIPLUS_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    aiplus_terms_rows.append((t, b, o, b - o))
    if len(aiplus_terms_rows) >= TOP_N_AIPLUS_TERMS:
        break

aiplus_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in aiplus_terms_rows]
brics_vals = [r[1] for r in aiplus_terms_rows]
ocde_vals = [r[2] for r in aiplus_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(aiplus_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(aiplus_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo AI+: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 4.3. Mapa combinado — PBIA e AI+ juntos entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa reaproveita o mapa da Seção 3.3, mas
agora posiciona **os dois documentos nacionais ao mesmo tempo** — PBIA
(Brasil) e AI+ (China) — permitindo comparar diretamente até que ponto cada
país se aproxima mais do eixo de desenvolvimento do BRICS ou do eixo de
governança da OCDE. Como antes, cada documento aparece com dois pontos
(cosseno, o principal, e Jaccard, o secundário, ligados por uma linha
pontilhada), e os cantos de referência marcam onde o próprio BRICS e a
própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)
ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# AI+ — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_aiplus_brics, cosine_aiplus_brics], [jaccard_aiplus_ocde, cosine_aiplus_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_AIPLUS, zorder=4)
ax.scatter([jaccard_aiplus_brics], [jaccard_aiplus_ocde], s=110, color=CATEGORICAL_AIPLUS, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="AI+ (Jaccard)")
ax.scatter([cosine_aiplus_brics], [cosine_aiplus_ocde], s=340, color=CATEGORICAL_AIPLUS, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="AI+ (cosseno)")
ax.annotate("AI+", (cosine_aiplus_brics, cosine_aiplus_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA e AI+ no mapa BRICS × OCDE — qual país se aproxima de qual eixo?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()


## Síntese — AI+, PBIA, BRICS e OCDE

- Nas duas métricas, o **AI+ também se aproxima mais do eixo do BRICS do que
  do eixo da OCDE**: similaridade de cosseno de **0,417** com o BRICS contra
  **0,330** com a OCDE, e Jaccard de **0,190** contra **0,155**. O padrão
  repete o observado no PBIA (0,538 × 0,395), embora com similaridade
  absoluta mais baixa nos dois eixos — o AI+ é, dos três documentos
  analisados, o que menos se parece tanto com o BRICS quanto com a OCDE.
- A similaridade do AI+ com o **próprio PBIA** é notavelmente alta: cosseno
  de **0,404** (quase empatado com o BRICS) e Jaccard de **0,205** — o maior
  Jaccard entre os três pares do AI+. Ou seja, em termos de sobreposição
  bruta de vocabulário, o AI+ se parece mais com o plano brasileiro do que
  com a própria declaração do BRICS, mesmo não integrando o bloco.
- O vocabulário mais usado pelo próprio AI+ confirma essa inclinação: termos
  centrais como *development* (32 ocorrências) e *governance* (10) aparecem
  proporcionalmente mais no BRICS do que na OCDE, enquanto termos como
  *systems* e *research* pendem para o lado da OCDE — mas em volume bem
  menor. Muito do vocabulário mais frequente do AI+ (*intelligent*,
  *application*, *smart*, *terminal*, *agents*) é técnico e específico da
  política chinesa, sem equivalente forte em nenhum dos dois documentos de
  referência — o que explica por que as similaridades absolutas do AI+ são
  mais baixas que as do PBIA.
- No **mapa combinado**, tanto o PBIA quanto o AI+ ficam do lado do eixo X
  (mais perto do BRICS do que da OCDE), mas o PBIA está bem mais próximo do
  canto de referência do BRICS do que o AI+. O plano brasileiro absorve mais
  explicitamente a linguagem de desenvolvimento e cooperação do BRICS,
  enquanto a política chinesa, apesar de também pender para esse eixo,
  mantém um vocabulário técnico e de infraestrutura de IA mais próprio, que
  não é plenamente capturado nem pelo eixo do BRICS nem pelo da OCDE.

## 5. Posicionamento do Brasil (PBIA) diante do BRICS, da OCDE e do AI+

As seções anteriores compararam os documentos dois a dois. Esta seção reúne
as seis similaridades de cosseno já calculadas (BRICS×OCDE, BRICS×AI+,
BRICS×PBIA, OCDE×AI+, OCDE×PBIA, AI+×PBIA) em duas visualizações de síntese,
de leitura acadêmica consolidada:

1. **Gráfico radar** — o perfil de similaridade do PBIA nos três eixos
   (BRICS, OCDE, AI+), comparado a uma linha de referência: a similaridade
   *média* que BRICS, OCDE e AI+ têm entre si. Isso responde: o Brasil está
   mais ou menos alinhado a cada bloco do que os próprios blocos
   internacionais estão entre si?
2. **Mapa de escalonamento multidimensional (MDS clássico)** — projeta os
   quatro documentos em um plano 2D a partir da matriz completa de
   distâncias (1 − similaridade de cosseno), preservando o quanto possível
   as distâncias relativas originais. É a técnica padrão em linguística de
   corpus e cientometria para visualizar espaços de similaridade textual.

### 5.1. Gráfico radar — perfil de alinhamento do PBIA

Cada eixo é a similaridade de cosseno do PBIA com um documento de
referência. A linha tracejada cinza é a **similaridade média entre os
próprios três documentos internacionais** nesse mesmo eixo (ex.: no eixo
BRICS, a média de OCDE×BRICS e AI+×BRICS) — o "quanto os blocos
internacionais se parecem entre si", usado como referência de comparação
para o perfil do Brasil.

In [ ]:
import numpy as np

radar_categories = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL]

pbia_profile = [cosine_pbia_brics, cosine_pbia_ocde, cosine_aiplus_pbia]

# Referência: para cada eixo, a média de similaridade entre os OUTROS dois
# documentos internacionais (ex.: no eixo BRICS, a média de OCDE×BRICS e AI+×BRICS)
international_baseline = [
    (cosine_similarity + cosine_aiplus_brics) / 2,   # eixo BRICS: OCDE×BRICS e AI+×BRICS
    (cosine_similarity + cosine_aiplus_ocde) / 2,    # eixo OCDE: BRICS×OCDE e AI+×OCDE
    (cosine_aiplus_brics + cosine_aiplus_ocde) / 2,  # eixo AI+: BRICS×AI+ e OCDE×AI+
]

print("Perfil PBIA:            ", [f"{v:.3f}" for v in pbia_profile])
print("Referência internacional:", [f"{v:.3f}" for v in international_baseline])

n_axes = len(radar_categories)
angles = [n / float(n_axes) * 2 * np.pi for n in range(n_axes)]
angles += angles[:1]

pbia_plot = pbia_profile + pbia_profile[:1]
baseline_plot = international_baseline + international_baseline[:1]

fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(polar=True), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_categories, color=INK_PRIMARY, fontsize=11)

max_r = max(pbia_plot + baseline_plot) * 1.25
ax.set_ylim(0, max_r)
r_ticks = np.round(np.linspace(0, max_r, 5), 2)
ax.set_yticks(r_ticks)
ax.set_yticklabels([f"{r:.2f}" for r in r_ticks], color=INK_MUTED, fontsize=8)
ax.tick_params(axis="x", pad=12)

ax.spines["polar"].set_color(GRIDLINE)
ax.grid(color=GRIDLINE, linewidth=0.8)

ax.plot(angles, baseline_plot, linestyle="--", linewidth=1.4, color=INK_MUTED,
        label="Média entre os 3 documentos internacionais")
ax.fill(angles, baseline_plot, color=INK_MUTED, alpha=0.06)

ax.plot(angles, pbia_plot, linewidth=2.2, color=CATEGORICAL_PBIA, label=PBIA_LABEL)
ax.fill(angles, pbia_plot, color=CATEGORICAL_PBIA, alpha=0.18)
ax.scatter(angles, pbia_plot, s=50, color=CATEGORICAL_PBIA, zorder=5, edgecolors=SURFACE, linewidths=1.0)

for angle, value in zip(angles, pbia_plot):
    ax.annotate(f"{value:.3f}", (angle, value), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9, color=INK_PRIMARY, fontweight="bold")

ax.set_title(
    "Perfil de alinhamento do PBIA — similaridade de cosseno com BRICS, OCDE e AI+",
    color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=28,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 5.2. Mapa de escalonamento multidimensional (MDS) — os quatro documentos no mesmo plano

Construímos a matriz completa de distâncias entre os quatro documentos
(distância = 1 − similaridade de cosseno) e aplicamos o **MDS clássico**
(Torgerson): dupla centralização da matriz de distâncias ao quadrado,
seguida de decomposição em autovalores/autovetores, mantendo os dois
componentes de maior variância. O resultado é um mapa 2D em que a distância
entre dois pontos aproxima, o melhor possível, a dissimilaridade textual
real entre os dois documentos — sem impor de antemão nenhum eixo temático
(diferente do mapa BRICS × OCDE das seções 3.3/4.3, aqui nenhum documento é
usado como eixo de referência).

In [ ]:
doc_labels = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL]
doc_colors = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA)
S = np.array([
    [1.0,                cosine_similarity,     cosine_aiplus_brics, cosine_pbia_brics],
    [cosine_similarity,  1.0,                   cosine_aiplus_ocde,  cosine_pbia_ocde],
    [cosine_aiplus_brics, cosine_aiplus_ocde,   1.0,                 cosine_aiplus_pbia],
    [cosine_pbia_brics,  cosine_pbia_ocde,      cosine_aiplus_pbia,  1.0],
])
D = 1.0 - S
np.fill_diagonal(D, 0.0)

def classical_mds(dist_matrix, n_components=2):
    n = dist_matrix.shape[0]
    D2 = dist_matrix ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J
    eigvals, eigvecs = np.linalg.eigh(B)
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    eigvals_pos = np.clip(eigvals, 0, None)
    coords = eigvecs[:, :n_components] * np.sqrt(eigvals_pos[:n_components])
    explained = eigvals_pos[:n_components].sum() / eigvals_pos.sum()
    return coords, explained

coords, explained_variance = classical_mds(D)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels, coords):
    print(f"  {label:24s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance*100:.1f}%")

fig, ax = plt.subplots(figsize=(7.5, 7.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels)):
    for j in range(i + 1, len(doc_labels)):
        sim = S[i, j]
        ax.plot([coords[i, 0], coords[j, 0]], [coords[i, 1], coords[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels, doc_colors, coords):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords).max() * 0.45, 0.05)
ax.set_xlim(coords[:, 0].min() - pad, coords[:, 0].max() + pad)
ax.set_ylim(coords[:, 1].min() - pad, coords[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos quatro documentos (MDS clássico, {explained_variance*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### Síntese — o posicionamento do Brasil

- No **gráfico radar**, o PBIA supera a similaridade média internacional
  nos eixos **BRICS** (0,538 vs. 0,468) e **AI+** (0,404 vs. 0,374), e fica
  ligeiramente abaixo dela apenas no eixo **OCDE** (0,395 vs. 0,425). Ou
  seja, o Brasil está **mais próximo do BRICS e do AI+ do que os documentos
  internacionais estão, em média, uns dos outros**, e apenas discretamente
  mais distante da OCDE do que a média do próprio grupo internacional — um
  desalinhamento pequeno, não um afastamento acentuado.
- No **mapa MDS**, a ligação **BRICS–PBIA (cosseno 0,538)** é o par mais
  próximo de todo o conjunto de quatro documentos — mais próximo até do que
  o par BRICS–OCDE (0,519), que serviu de referência nas seções anteriores.
  O BRICS funciona como um "elo": é o documento mais parecido tanto com o
  PBIA quanto com a OCDE. Já o par **OCDE–AI+ (0,330)** é o mais distante de
  todo o estudo — a política chinesa e a recomendação da OCDE são, em termos
  de vocabulário e ênfase, os dois documentos mais diferentes entre si.
- O PBIA fica claramente puxado para perto do BRICS no mapa, enquanto suas
  distâncias até a OCDE (0,605) e até o AI+ (0,583) são parecidas entre
  si — ou seja, o Brasil não está simplesmente "afastado da OCDE": ele tem
  uma afinidade específica e mensurável com o BRICS, e uma distância
  semelhante tanto da OCDE quanto do AI+, os dois documentos mais técnicos
  e "institucionais" do conjunto.
- Em conjunto, as duas visualizações sustentam a mesma conclusão por
  caminhos metodológicos independentes (perfil radial vs. projeção
  geométrica de distâncias, que preservou fielmente a ordem de todas as
  seis distâncias par a par): a política brasileira de IA (PBIA) se
  posiciona, na sua ênfase textual, mais próxima da agenda de
  desenvolvimento tecnológico nacional do BRICS do que de qualquer outro
  documento do estudo — inclusive do AI+, apesar de este também pender
  para o eixo BRICS quando comparado isoladamente com a OCDE.

## 6. New Generation AI Development Plan (China New Gen) — a qual documento o plano chinês se aproxima mais?

O **China New Gen** (`China_New_Gen.json`) é o *State Council Notice on the
Issuance of the New Generation Artificial Intelligence Development Plan*
(2017), o plano-mestre de longo prazo da China para IA — anterior e mais
amplo que o AI+ (que é uma política setorial de 2023 de aprofundamento da
implementação). Diferente dos demais documentos, o JSON deste plano guarda o
texto em uma árvore (`sections` → capítulos → subseções → itens numerados →
boxes, cada nível com `title` e `paragraphs`), em vez de uma lista simples de
seções com campo `text`. Por isso, ele usa uma função de carregamento própria
que percorre essa árvore recursivamente, mas aplica exatamente o mesmo filtro
de tokenização (mesma lista de *stopwords*, mesmo filtro de tamanho mínimo de
palavra) usado para BRICS, OCDE, PBIA e AI+, para que todas as séries de
frequência continuem diretamente comparáveis.

Comparamos o China New Gen com os quatro documentos já carregados — BRICS
Declaration, OECD Recommendation, PBIA e AI+ — para responder: o vocabulário
e a ênfase temática do plano chinês de longo prazo se aproximam mais do eixo
**desenvolvimento / países em desenvolvimento / economia digital** (BRICS),
do eixo **arquitetura de governança / atores responsáveis** (OCDE), do plano
brasileiro (PBIA), ou da política setorial mais recente da própria China
(AI+)?

In [ ]:
CHINA_LABEL = "China New Gen"

def load_tokens_nested(path):
    """Carrega tokens de um JSON com texto em árvore (sections -> capítulos ->
    subseções -> itens -> boxes), como o China_New_Gen.json, em vez da lista
    simples de seções com campo "text" usada pelos demais documentos."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    parts = [data["document"]["title"], data["document"].get("preamble", "")]

    def walk(node, parts):
        if "title" in node:
            parts.append(node["title"])
        if "paragraphs" in node:
            parts.extend(node["paragraphs"])
        for key in ("subsections", "items", "boxes"):
            for child in node.get(key, []):
                walk(child, parts)

    for chapter in data["sections"]:
        walk(chapter, parts)

    text = " ".join(parts)
    tokens = re.findall(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2], data

china_tokens, china_data = load_tokens_nested("China_New_Gen.json")
china_counts = Counter(china_tokens)

print(f"{CHINA_LABEL}: {len(china_tokens)} tokens, {len(china_counts)} termos únicos")


### 6.1. Similaridade do China New Gen com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (China New Gen, BRICS), (China New Gen,
OCDE), (China New Gen, PBIA) e (China New Gen, AI+).

In [ ]:
jaccard_china_brics, cosine_china_brics = jaccard_cosine(china_counts, china_tokens, brics_counts, brics_tokens)
jaccard_china_ocde, cosine_china_ocde = jaccard_cosine(china_counts, china_tokens, ocde_counts, ocde_tokens)
jaccard_china_pbia, cosine_china_pbia = jaccard_cosine(china_counts, china_tokens, pbia_counts, pbia_tokens)
jaccard_china_aiplus, cosine_china_aiplus = jaccard_cosine(china_counts, china_tokens, aiplus_counts, aiplus_tokens)

print(f"{CHINA_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_brics:.3f}")
print()
print(f"{CHINA_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_ocde:.3f}")
print()
print(f"{CHINA_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_pbia:.3f}")
print()
print(f"{CHINA_LABEL} × {AIPLUS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_china_aiplus:.3f}")
print(f"  Similaridade de cosseno: {cosine_china_aiplus:.3f}")


### Gráfico — com qual documento o China New Gen se parece mais?

As quatro comparações lado a lado (China New Gen × BRICS, × OCDE, × PBIA,
× AI+), para as duas métricas.

In [ ]:
CATEGORICAL_CHINA = "#d6336c"  # cor de identidade do China New Gen nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_china_brics, cosine_china_brics]
ocde_values = [jaccard_china_ocde, cosine_china_ocde]
pbia_values = [jaccard_china_pbia, cosine_china_pbia]
aiplus_values = [jaccard_china_aiplus, cosine_china_aiplus]

fig, ax = plt.subplots(figsize=(8, 6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.18
offsets = [1.5 * bar_h + 0.03, 0.5 * bar_h + 0.01, -(0.5 * bar_h + 0.01), -(1.5 * bar_h + 0.03)]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{CHINA_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{CHINA_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{CHINA_LABEL} × {PBIA_LABEL}"),
    (aiplus_values, offsets[3], CATEGORICAL_AIPLUS, f"{CHINA_LABEL} × {AIPLUS_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values + aiplus_values
ax.set_xlim(0, max(all_values) * 1.35)
ax.set_title("Com qual documento o China New Gen se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 6.2. O vocabulário mais usado pelo China New Gen ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do China New Gen (excluindo palavras
autorreferentes como *china* e *chinese*), comparamos o quanto **BRICS** e
**OCDE** usam cada um desses mesmos termos — a mesma análise feita para o
PBIA na Seção 3.2 e para o AI+ na Seção 4.2, agora para o plano-mestre
chinês de longo prazo.

In [ ]:
SELF_REFERENTIAL_CHINA = {"china", "chinese"}
MIN_CHINA_COUNT = 3
TOP_N_CHINA_TERMS = 18

china_terms_rows = []
for t, china_c in china_counts.most_common():
    if t in SELF_REFERENTIAL_CHINA or china_c < MIN_CHINA_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    china_terms_rows.append((t, b, o, b - o))
    if len(china_terms_rows) >= TOP_N_CHINA_TERMS:
        break

china_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in china_terms_rows]
brics_vals = [r[1] for r in china_terms_rows]
ocde_vals = [r[2] for r in china_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(china_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(china_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo China New Gen: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 6.3. Mapa combinado — PBIA, AI+ e China New Gen juntos entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa estende o da Seção 4.3, agora posicionando
os **três documentos nacionais/de longo prazo ao mesmo tempo** — PBIA
(Brasil), AI+ (China, política setorial de 2023) e China New Gen (China,
plano-mestre de 2017) — no mesmo eixo BRICS × OCDE. Como antes, cada
documento aparece com dois pontos (cosseno, o principal, e Jaccard, o
secundário, ligados por uma linha pontilhada), e os cantos de referência
marcam onde o próprio BRICS e a própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)
ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# AI+ — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_aiplus_brics, cosine_aiplus_brics], [jaccard_aiplus_ocde, cosine_aiplus_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_AIPLUS, zorder=4)
ax.scatter([jaccard_aiplus_brics], [jaccard_aiplus_ocde], s=110, color=CATEGORICAL_AIPLUS, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="AI+ (Jaccard)")
ax.scatter([cosine_aiplus_brics], [cosine_aiplus_ocde], s=340, color=CATEGORICAL_AIPLUS, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="AI+ (cosseno)")
ax.annotate("AI+", (cosine_aiplus_brics, cosine_aiplus_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# China New Gen — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_china_brics, cosine_china_brics], [jaccard_china_ocde, cosine_china_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_CHINA, zorder=4)
ax.scatter([jaccard_china_brics], [jaccard_china_ocde], s=110, color=CATEGORICAL_CHINA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="China New Gen (Jaccard)")
ax.scatter([cosine_china_brics], [cosine_china_ocde], s=340, color=CATEGORICAL_CHINA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="China New Gen (cosseno)")
ax.annotate("China New Gen", (cosine_china_brics, cosine_china_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA, AI+ e China New Gen no mapa BRICS × OCDE — qual país/plano se aproxima de qual eixo?",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()


### Síntese — China New Gen, BRICS, OCDE, PBIA e AI+

- O China New Gen também pende para o eixo do **BRICS** em vez do eixo da
  **OCDE**, replicando o padrão observado no PBIA e no AI+ — os três
  documentos nacionais do estudo se aproximam mais da ênfase em
  desenvolvimento tecnológico do BRICS do que da arquitetura de governança
  da OCDE.
- A maior similaridade do China New Gen, no entanto, é com o **AI+**: os
  dois documentos são políticas chinesas de IA (separadas por seis anos, o
  plano-mestre de 2017 e o aprofundamento setorial de 2023) e compartilham
  vocabulário técnico específico — *intelligent*, *smart*, *technology*,
  *innovation*, *systems* — não replicado nos demais documentos do estudo.
- No **mapa combinado**, o China New Gen aparece deslocado dos outros dois
  pontos nacionais (PBIA e AI+), mais próximo da diagonal BRICS=OCDE — sinal
  de que, apesar de também pender para o BRICS, o plano chinês de longo
  prazo tem uma similaridade absoluta menor com os dois blocos do que PBIA e
  AI+, refletindo seu vocabulário mais técnico e menos político/diplomático
  (o texto é dominado por metas de P&D, infraestrutura e insdústria, não por
  linguagem de cooperação internacional ou governança).

## 7. Síntese final — os cinco documentos juntos

Estendemos o mapa de escalonamento multidimensional (MDS) da Seção 5.2 para
incluir o **China New Gen** como um quinto documento, ao lado de BRICS, OCDE,
AI+ e PBIA. Construímos a matriz completa de distâncias entre os cinco
documentos (distância = 1 − similaridade de cosseno) e reaproveitamos a
função `classical_mds` já definida na Seção 5.2, mantendo os dois componentes
de maior variância — sem impor de antemão nenhum eixo temático, ao contrário
dos mapas BRICS × OCDE das seções 4.3/6.3.

In [ ]:
doc_labels5 = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL, CHINA_LABEL]
doc_colors5 = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA, CATEGORICAL_CHINA]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA, China New Gen)
S5 = np.array([
    [1.0,                 cosine_similarity,    cosine_aiplus_brics, cosine_pbia_brics, cosine_china_brics],
    [cosine_similarity,   1.0,                  cosine_aiplus_ocde,  cosine_pbia_ocde,  cosine_china_ocde],
    [cosine_aiplus_brics, cosine_aiplus_ocde,   1.0,                 cosine_aiplus_pbia, cosine_china_aiplus],
    [cosine_pbia_brics,   cosine_pbia_ocde,     cosine_aiplus_pbia,  1.0,                cosine_china_pbia],
    [cosine_china_brics,  cosine_china_ocde,    cosine_china_aiplus, cosine_china_pbia,  1.0],
])
D5 = 1.0 - S5
np.fill_diagonal(D5, 0.0)

coords5, explained_variance5 = classical_mds(D5)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels5, coords5):
    print(f"  {label:24s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance5*100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels5)):
    for j in range(i + 1, len(doc_labels5)):
        sim = S5[i, j]
        ax.plot([coords5[i, 0], coords5[j, 0]], [coords5[i, 1], coords5[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels5, doc_colors5, coords5):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords5).max() * 0.45, 0.05)
ax.set_xlim(coords5[:, 0].min() - pad, coords5[:, 0].max() + pad)
ax.set_ylim(coords5[:, 1].min() - pad, coords5[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos cinco documentos (MDS clássico, {explained_variance5*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### Síntese — o mapa completo dos cinco documentos

- A ligação mais forte de **todo** o mapa de cinco documentos passa a ser
  **China New Gen–AI+ (cosseno 0,742)** — muito acima de qualquer par
  observado até aqui, inclusive do antigo campeão **BRICS–PBIA (0,538)**, que
  era o par mais próximo antes da entrada do China New Gen (Seção 5.2). Isso
  evidencia que a proximidade de autoria — duas políticas chinesas de IA,
  mesmo separadas por seis anos e por escopos diferentes — supera qualquer
  proximidade de bloco político observada no restante do estudo, e no mapa
  aparece como o par de pontos visivelmente mais próximo entre si.
- Logo atrás, o restante da hierarquia de proximidade observada na Seção 5
  se mantém: **BRICS–PBIA (0,538)** e **BRICS–OCDE (0,519)** seguem como o
  segundo e o terceiro pares mais próximos, com o **BRICS permanecendo o
  "elo"** do conjunto — o documento mais parecido com praticamente todos os
  outros.
- O par mais distante de todo o mapa continua sendo **AI+–OCDE (0,330)**,
  seguido de perto por **China New Gen–OCDE (0,379)** — as duas políticas
  chinesas são, das cinco, as que menos dialogam com a linguagem de
  governança, atores e responsabilidade da OCDE.
- O **China New Gen** ocupa uma posição própria no mapa, junto ao AI+ (dado
  o par mais forte do estudo) mas puxado para mais perto do centro do que
  ele — reflexo de uma similaridade absoluta um pouco maior com BRICS e PBIA
  do que a que o AI+ tem com os mesmos dois documentos (0,447 e 0,475 do
  China New Gen contra 0,417 e 0,404 do AI+).

## 8. Apply AI Strategy (UE) — a qual documento a estratégia europeia se aproxima mais?

O **Apply AI Strategy** (`Apply_AI_Strategy.json`) é a comunicação da Comissão
Europeia *Apply AI Strategy* (COM(2025) 723 final, out/2025) — a estratégia da
UE para acelerar a adoção setorial de IA (saúde, manufatura, defesa,
mobilidade, setor público, etc.). Assim como BRICS, OCDE, PBIA e AI+, este
JSON guarda o texto em uma lista simples de seções com campo `text`, então
reaproveitamos diretamente a função `load_tokens` da Seção 1 (mesma lista de
*stopwords*, mesmo filtro de tamanho mínimo de palavra).

Comparamos o Apply AI Strategy com os cinco documentos já carregados — BRICS
Declaration, OECD Recommendation, PBIA, AI+ e China New Gen — para responder:
sendo a UE um bloco de países desenvolvidos com forte tradição regulatória,
o vocabulário e a ênfase temática do Apply AI Strategy se aproximam mais do
eixo **arquitetura de governança / atores responsáveis** da OCDE, ou do eixo
**desenvolvimento / economia digital** do BRICS — invertendo o padrão visto
em PBIA, AI+ e China New Gen, os três documentos que até aqui penderam para
o BRICS?

In [ ]:
APPLY_LABEL = "Apply AI Strategy (UE)"

apply_tokens, apply_data = load_tokens("Apply_AI_Strategy.json")
apply_counts = Counter(apply_tokens)

print(f"{APPLY_LABEL}: {len(apply_tokens)} tokens, {len(apply_counts)} termos únicos")


### 8.1. Similaridade do Apply AI Strategy com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (Apply AI Strategy, BRICS), (Apply AI
Strategy, OCDE), (Apply AI Strategy, PBIA), (Apply AI Strategy, AI+) e
(Apply AI Strategy, China New Gen).

In [ ]:
jaccard_apply_brics, cosine_apply_brics = jaccard_cosine(apply_counts, apply_tokens, brics_counts, brics_tokens)
jaccard_apply_ocde, cosine_apply_ocde = jaccard_cosine(apply_counts, apply_tokens, ocde_counts, ocde_tokens)
jaccard_apply_pbia, cosine_apply_pbia = jaccard_cosine(apply_counts, apply_tokens, pbia_counts, pbia_tokens)
jaccard_apply_aiplus, cosine_apply_aiplus = jaccard_cosine(apply_counts, apply_tokens, aiplus_counts, aiplus_tokens)
jaccard_apply_china, cosine_apply_china = jaccard_cosine(apply_counts, apply_tokens, china_counts, china_tokens)

print(f"{APPLY_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_brics:.3f}")
print()
print(f"{APPLY_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_ocde:.3f}")
print()
print(f"{APPLY_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_pbia:.3f}")
print()
print(f"{APPLY_LABEL} × {AIPLUS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_aiplus:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_aiplus:.3f}")
print()
print(f"{APPLY_LABEL} × {CHINA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_apply_china:.3f}")
print(f"  Similaridade de cosseno: {cosine_apply_china:.3f}")


### Gráfico — com qual documento o Apply AI Strategy se parece mais?

As cinco comparações lado a lado (Apply AI Strategy × BRICS, × OCDE, × PBIA,
× AI+, × China New Gen), para as duas métricas.

In [ ]:
CATEGORICAL_APPLY = "#f59f00"  # cor de identidade do Apply AI Strategy nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_apply_brics, cosine_apply_brics]
ocde_values = [jaccard_apply_ocde, cosine_apply_ocde]
pbia_values = [jaccard_apply_pbia, cosine_apply_pbia]
aiplus_values = [jaccard_apply_aiplus, cosine_apply_aiplus]
china_values = [jaccard_apply_china, cosine_apply_china]

fig, ax = plt.subplots(figsize=(8, 6.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.15
offsets = [2 * bar_h + 0.04, bar_h + 0.02, 0.0, -(bar_h + 0.02), -(2 * bar_h + 0.04)]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{APPLY_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{APPLY_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{APPLY_LABEL} × {PBIA_LABEL}"),
    (aiplus_values, offsets[3], CATEGORICAL_AIPLUS, f"{APPLY_LABEL} × {AIPLUS_LABEL}"),
    (china_values, offsets[4], CATEGORICAL_CHINA, f"{APPLY_LABEL} × {CHINA_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values + aiplus_values + china_values
ax.set_xlim(0, max(all_values) * 1.35)
ax.set_title("Com qual documento o Apply AI Strategy se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 8.2. O vocabulário mais usado pelo Apply AI Strategy ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do Apply AI Strategy (excluindo palavras
autorreferentes como *european*, *europe*, *commission* e *eu*), comparamos o
quanto **BRICS** e **OCDE** usam cada um desses mesmos termos — a mesma
análise feita para PBIA, AI+ e China New Gen nas seções anteriores, agora
para a estratégia europeia.

In [ ]:
SELF_REFERENTIAL_APPLY = {"european", "europe", "commission", "eu"}
MIN_APPLY_COUNT = 3
TOP_N_APPLY_TERMS = 18

apply_terms_rows = []
for t, apply_c in apply_counts.most_common():
    if t in SELF_REFERENTIAL_APPLY or apply_c < MIN_APPLY_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    apply_terms_rows.append((t, b, o, b - o))
    if len(apply_terms_rows) >= TOP_N_APPLY_TERMS:
        break

apply_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in apply_terms_rows]
brics_vals = [r[1] for r in apply_terms_rows]
ocde_vals = [r[2] for r in apply_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(apply_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(apply_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo Apply AI Strategy: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 8.3. Mapa combinado — Apply AI Strategy junto ao China New Gen, AI+ e PBIA entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa estende o da Seção 6.3, agora posicionando
os **quatro documentos nacionais/de bloco ao mesmo tempo** — PBIA (Brasil),
AI+ (China, 2023), China New Gen (China, 2017) e Apply AI Strategy (UE,
2025) — no mesmo eixo BRICS × OCDE. Como antes, cada documento aparece com
dois pontos (cosseno, o principal, e Jaccard, o secundário, ligados por uma
linha pontilhada), e os cantos de referência marcam onde o próprio BRICS e a
própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

# PBIA — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_pbia_brics, cosine_pbia_brics], [jaccard_pbia_ocde, cosine_pbia_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_PBIA, zorder=4)
ax.scatter([jaccard_pbia_brics], [jaccard_pbia_ocde], s=110, color=CATEGORICAL_PBIA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="PBIA (Jaccard)")
ax.scatter([cosine_pbia_brics], [cosine_pbia_ocde], s=340, color=CATEGORICAL_PBIA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="PBIA (cosseno)")
ax.annotate("PBIA", (cosine_pbia_brics, cosine_pbia_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# AI+ — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_aiplus_brics, cosine_aiplus_brics], [jaccard_aiplus_ocde, cosine_aiplus_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_AIPLUS, zorder=4)
ax.scatter([jaccard_aiplus_brics], [jaccard_aiplus_ocde], s=110, color=CATEGORICAL_AIPLUS, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="AI+ (Jaccard)")
ax.scatter([cosine_aiplus_brics], [cosine_aiplus_ocde], s=340, color=CATEGORICAL_AIPLUS, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="AI+ (cosseno)")
ax.annotate("AI+", (cosine_aiplus_brics, cosine_aiplus_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# China New Gen — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_china_brics, cosine_china_brics], [jaccard_china_ocde, cosine_china_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_CHINA, zorder=4)
ax.scatter([jaccard_china_brics], [jaccard_china_ocde], s=110, color=CATEGORICAL_CHINA, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="China New Gen (Jaccard)")
ax.scatter([cosine_china_brics], [cosine_china_ocde], s=340, color=CATEGORICAL_CHINA, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="China New Gen (cosseno)")
ax.annotate("China New Gen", (cosine_china_brics, cosine_china_ocde), textcoords="offset points", xytext=(10, -14),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

# Apply AI Strategy — ponto principal (cosseno) e ponto secundário (Jaccard)
ax.plot([jaccard_apply_brics, cosine_apply_brics], [jaccard_apply_ocde, cosine_apply_ocde],
        linestyle=":", linewidth=1.0, color=CATEGORICAL_APPLY, zorder=4)
ax.scatter([jaccard_apply_brics], [jaccard_apply_ocde], s=110, color=CATEGORICAL_APPLY, alpha=0.45,
           edgecolors=SURFACE, linewidths=0.8, zorder=5, label="Apply AI Strategy (Jaccard)")
ax.scatter([cosine_apply_brics], [cosine_apply_ocde], s=340, color=CATEGORICAL_APPLY, alpha=0.92,
           edgecolors=SURFACE, linewidths=1.2, zorder=6, label="Apply AI Strategy (cosseno)")
ax.annotate("Apply AI Strategy", (cosine_apply_brics, cosine_apply_ocde), textcoords="offset points", xytext=(10, 10),
            fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA, AI+, China New Gen e Apply AI Strategy no mapa BRICS × OCDE",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()


### Síntese — Apply AI Strategy, BRICS, OCDE, PBIA, AI+ e China New Gen

- Contra a hipótese natural de que uma estratégia da Comissão Europeia
  penderia para o eixo de governança da OCDE, o resultado é o **oposto**: a
  **OCDE é o documento com que o Apply AI Strategy menos se parece**
  (cosseno 0,375) — abaixo até da similaridade com o AI+ chinês (0,398). O
  Apply AI Strategy se aproxima mais, em ordem, do **PBIA** (0,487), do
  **BRICS** (0,443), do **China New Gen** (0,429) e só depois do AI+ (0,398)
  e da OCDE (0,375, o menor de todos).
- O padrão se repete de forma notável: **os quatro documentos "nacionais"
  analisados — PBIA, AI+, China New Gen e Apply AI Strategy — penderam,
  todos, para o eixo do BRICS em vez do eixo da OCDE**, com margens de
  0,143 (PBIA), 0,087 (AI+), 0,068 (China New Gen) e 0,068 (Apply AI
  Strategy, empatado com o China New Gen). Nenhum dos quatro planos
  nacionais/regionais de IA se aproxima mais da OCDE do que do BRICS.
- Uma explicação plausível: PBIA, AI+, China New Gen e Apply AI Strategy são
  todos **planos estratégicos multissetoriais** organizados por setor, com
  ações, metas e investimentos concretos — compartilhando vocabulário
  estrutural (*strategy*, *sectors*, *support*, *development*, *innovation*)
  que os aproxima entre si e do BRICS (também rico em vocabulário de
  desenvolvimento e cooperação tecnológica). A OCDE, por sua vez, é uma
  recomendação de princípios de governança (*actors*, *stakeholders*,
  *lifecycle*, *trustworthy*), um registro linguístico distinto de qualquer
  um dos quatro planos nacionais — inclusive do europeu.

## 9. Síntese final — os seis documentos juntos

Estendemos o mapa de escalonamento multidimensional (MDS) da Seção 7 para
incluir o **Apply AI Strategy** como um sexto documento, ao lado de BRICS,
OCDE, AI+, PBIA e China New Gen. Construímos a matriz completa de distâncias
entre os seis documentos (distância = 1 − similaridade de cosseno) e
reaproveitamos a função `classical_mds` já definida na Seção 5.2, mantendo os
dois componentes de maior variância.

In [ ]:
doc_labels6 = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL, CHINA_LABEL, APPLY_LABEL]
doc_colors6 = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA, CATEGORICAL_CHINA, CATEGORICAL_APPLY]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA, China New Gen, Apply AI Strategy)
S6 = np.array([
    [1.0,                 cosine_similarity,    cosine_aiplus_brics, cosine_pbia_brics, cosine_china_brics, cosine_apply_brics],
    [cosine_similarity,   1.0,                  cosine_aiplus_ocde,  cosine_pbia_ocde,  cosine_china_ocde,  cosine_apply_ocde],
    [cosine_aiplus_brics, cosine_aiplus_ocde,   1.0,                 cosine_aiplus_pbia, cosine_china_aiplus, cosine_apply_aiplus],
    [cosine_pbia_brics,   cosine_pbia_ocde,     cosine_aiplus_pbia,  1.0,                cosine_china_pbia,  cosine_apply_pbia],
    [cosine_china_brics,  cosine_china_ocde,    cosine_china_aiplus, cosine_china_pbia,  1.0,                cosine_apply_china],
    [cosine_apply_brics,  cosine_apply_ocde,    cosine_apply_aiplus, cosine_apply_pbia,  cosine_apply_china, 1.0],
])
D6 = 1.0 - S6
np.fill_diagonal(D6, 0.0)

coords6, explained_variance6 = classical_mds(D6)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels6, coords6):
    print(f"  {label:24s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance6*100:.1f}%")

fig, ax = plt.subplots(figsize=(8.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels6)):
    for j in range(i + 1, len(doc_labels6)):
        sim = S6[i, j]
        ax.plot([coords6[i, 0], coords6[j, 0]], [coords6[i, 1], coords6[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels6, doc_colors6, coords6):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords6).max() * 0.45, 0.05)
ax.set_xlim(coords6[:, 0].min() - pad, coords6[:, 0].max() + pad)
ax.set_ylim(coords6[:, 1].min() - pad, coords6[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos seis documentos (MDS clássico, {explained_variance6*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### Síntese — o mapa completo dos seis documentos

- A variância explicada pelos dois componentes cai para **66,5%** (ante
  80,3% com cinco documentos) — com seis documentos, o espaço de
  similaridade textual já não cabe tão bem em duas dimensões, então o mapa
  deve ser lido como um retrato aproximado da estrutura geral, não como
  distâncias exatas par a par.
- A ligação mais forte de todo o conjunto de seis documentos permanece
  **China New Gen–AI+ (0,742)**, seguida por **BRICS–PBIA (0,538)** e
  **BRICS–OCDE (0,519)** — a hierarquia observada nas seções anteriores não
  muda com a entrada do Apply AI Strategy.
- O achado mais importante desta seção é que a **OCDE se confirma como o
  documento mais isolado de todo o estudo**: suas quatro menores
  similaridades de todo o conjunto de 15 pares são, justamente, as que ela
  tem com os quatro planos nacionais/regionais — PBIA (0,395), China New Gen
  (0,379), Apply AI Strategy (0,375) e AI+ (0,330, a mais baixa de todas). A
  única conexão relevante da OCDE é com o **BRICS** (0,519) — as duas únicas
  "declarações de princípios" do conjunto, em contraste com os quatro planos
  de ação setoriais. No mapa, a OCDE aparece isolada no canto inferior
  esquerdo, afastada de todos os demais pontos.
- O Apply AI Strategy, por sua vez, aparece próximo do **PBIA** no mapa
  (ambos no quadrante superior) — reforçando que a proximidade entre os dois
  vem da estrutura setorial compartilhada dos planos, não de afinidade
  política de bloco. Já **AI+** e **China New Gen** seguem formando o par
  visualmente mais próximo de toda a figura, à direita, refletindo a mesma
  autoria por trás dos dois documentos.
- Em conjunto, o mapa de seis documentos sugere três agrupamentos: (1) as
  duas políticas chinesas (AI+ e China New Gen), (2) os planos setoriais de
  PBIA e Apply AI Strategy, com o BRICS funcionando como elo próximo de
  ambos, e (3) a OCDE, isolada como o único documento de governança pura do
  conjunto.

## 10. America's AI Action Plan (EUA) — a qual documento o plano americano se aproxima mais?

O **America's AI Action Plan** (`AMERICA_AI_ACTION_PLAN.json`) é o plano da
Casa Branca (julho de 2025) estruturado em três pilares — inovação,
infraestrutura e diplomacia/segurança internacional. Diferente dos demais
documentos, este JSON não guarda o texto em uma lista simples de seções com
campo `text` nem em uma árvore de `sections`/`subsections`/`items`; ele guarda
metadados do documento, uma introdução e uma lista de `pillars`, cada um com
`sections` que por sua vez têm `recommended_policy_actions`. Por isso, ele usa
uma função de carregamento própria (`load_tokens_america`), que percorre essa
estrutura (título, subtítulo, citação de abertura, introdução, e para cada
pilar/seção o título, o resumo e o texto de cada ação de política), mas aplica
exatamente o mesmo filtro de tokenização (mesma lista de *stopwords*, mesmo
filtro de tamanho mínimo de palavra) usado para os demais documentos, para que
todas as séries de frequência continuem diretamente comparáveis.

Comparamos o America's AI Action Plan com os seis documentos já carregados —
BRICS Declaration, OECD Recommendation, PBIA, AI+, China New Gen e Apply AI
Strategy — para responder: sendo os EUA a origem histórica da corrida
tecnológica que o próprio documento descreve, o vocabulário e a ênfase
temática do plano americano se aproximam mais do eixo **arquitetura de
governança / atores responsáveis** da OCDE, ou do eixo **desenvolvimento /
economia digital** do BRICS — replicando o padrão visto em PBIA, AI+, China
New Gen e Apply AI Strategy, os quatro documentos que até aqui penderam,
todos, para o BRICS?

In [ ]:
AMERICA_LABEL = "America's AI Action Plan (EUA)"

def load_tokens_america(path):
    """Carrega tokens do AMERICA_AI_ACTION_PLAN.json, cujo texto está organizado
    em document_metadata / introduction / pillars -> sections ->
    recommended_policy_actions, em vez da lista simples de seções com campo
    "text" (load_tokens) ou da árvore sections -> subsections -> items -> boxes
    (load_tokens_nested) usadas pelos demais documentos."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    meta = data["document_metadata"]
    parts = [meta["title"], meta["subtitle"], meta["opening_quote"]["text"]]

    intro = data["introduction"]
    parts.append(intro["summary"])
    parts.extend(intro["key_points"])
    parts.extend(intro["cross_cutting_principles"])

    for pillar in data["pillars"]:
        parts.append(pillar["title"])
        parts.append(pillar["summary"])
        for section in pillar["sections"]:
            parts.append(section["section_title"])
            parts.append(section["summary"])
            for action in section["recommended_policy_actions"]:
                parts.append(action["action"])

    text = " ".join(parts)
    tokens = re.findall(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2], data

america_tokens, america_data = load_tokens_america("AMERICA_AI_ACTION_PLAN.json")
america_counts = Counter(america_tokens)

print(f"{AMERICA_LABEL}: {len(america_tokens)} tokens, {len(america_counts)} termos únicos")

### 10.1. Similaridade do America's AI Action Plan com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (America's AI Action Plan, BRICS),
(America's AI Action Plan, OCDE), (America's AI Action Plan, PBIA),
(America's AI Action Plan, AI+), (America's AI Action Plan, China New Gen) e
(America's AI Action Plan, Apply AI Strategy).

In [ ]:
jaccard_america_brics, cosine_america_brics = jaccard_cosine(america_counts, america_tokens, brics_counts, brics_tokens)
jaccard_america_ocde, cosine_america_ocde = jaccard_cosine(america_counts, america_tokens, ocde_counts, ocde_tokens)
jaccard_america_pbia, cosine_america_pbia = jaccard_cosine(america_counts, america_tokens, pbia_counts, pbia_tokens)
jaccard_america_aiplus, cosine_america_aiplus = jaccard_cosine(america_counts, america_tokens, aiplus_counts, aiplus_tokens)
jaccard_america_china, cosine_america_china = jaccard_cosine(america_counts, america_tokens, china_counts, china_tokens)
jaccard_america_apply, cosine_america_apply = jaccard_cosine(america_counts, america_tokens, apply_counts, apply_tokens)

print(f"{AMERICA_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_america_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_america_brics:.3f}")
print()
print(f"{AMERICA_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_america_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_america_ocde:.3f}")
print()
print(f"{AMERICA_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_america_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_america_pbia:.3f}")
print()
print(f"{AMERICA_LABEL} × {AIPLUS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_america_aiplus:.3f}")
print(f"  Similaridade de cosseno: {cosine_america_aiplus:.3f}")
print()
print(f"{AMERICA_LABEL} × {CHINA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_america_china:.3f}")
print(f"  Similaridade de cosseno: {cosine_america_china:.3f}")
print()
print(f"{AMERICA_LABEL} × {APPLY_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_america_apply:.3f}")
print(f"  Similaridade de cosseno: {cosine_america_apply:.3f}")

### Gráfico — com qual documento o America's AI Action Plan se parece mais?

As seis comparações lado a lado (America's AI Action Plan × BRICS, × OCDE,
× PBIA, × AI+, × China New Gen, × Apply AI Strategy), para as duas
métricas.

In [ ]:
CATEGORICAL_AMERICA = "#008300"  # cor de identidade do America's AI Action Plan nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_america_brics, cosine_america_brics]
ocde_values = [jaccard_america_ocde, cosine_america_ocde]
pbia_values = [jaccard_america_pbia, cosine_america_pbia]
aiplus_values = [jaccard_america_aiplus, cosine_america_aiplus]
china_values = [jaccard_america_china, cosine_america_china]
apply_values = [jaccard_america_apply, cosine_america_apply]

fig, ax = plt.subplots(figsize=(8, 7), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.13
offsets = [2.5 * bar_h + 0.05, 1.5 * bar_h + 0.03, 0.5 * bar_h + 0.01,
           -(0.5 * bar_h + 0.01), -(1.5 * bar_h + 0.03), -(2.5 * bar_h + 0.05)]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{AMERICA_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{AMERICA_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{AMERICA_LABEL} × {PBIA_LABEL}"),
    (aiplus_values, offsets[3], CATEGORICAL_AIPLUS, f"{AMERICA_LABEL} × {AIPLUS_LABEL}"),
    (china_values, offsets[4], CATEGORICAL_CHINA, f"{AMERICA_LABEL} × {CHINA_LABEL}"),
    (apply_values, offsets[5], CATEGORICAL_APPLY, f"{AMERICA_LABEL} × {APPLY_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=8.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values + aiplus_values + china_values + apply_values
ax.set_xlim(0, max(all_values) * 1.4)
ax.set_title("Com qual documento o America's AI Action Plan se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()

### 10.2. O vocabulário mais usado pelo America's AI Action Plan ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do America's AI Action Plan (excluindo
palavras autorreferentes como *america*, *american*, *united*, *states* e
*trump*), comparamos o quanto **BRICS** e **OCDE** usam cada um desses mesmos
termos — a mesma análise feita para PBIA, AI+, China New Gen e Apply AI
Strategy nas seções anteriores, agora para o plano americano.

In [ ]:
SELF_REFERENTIAL_AMERICA = {"america", "american", "united", "states", "trump"}
MIN_AMERICA_COUNT = 3
TOP_N_AMERICA_TERMS = 18

america_terms_rows = []
for t, america_c in america_counts.most_common():
    if t in SELF_REFERENTIAL_AMERICA or america_c < MIN_AMERICA_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    america_terms_rows.append((t, b, o, b - o))
    if len(america_terms_rows) >= TOP_N_AMERICA_TERMS:
        break

america_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in america_terms_rows]
brics_vals = [r[1] for r in america_terms_rows]
ocde_vals = [r[2] for r in america_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(america_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(america_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo America's AI Action Plan: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()

### 10.3. Mapa combinado — America's AI Action Plan junto ao China New Gen, Apply AI Strategy, AI+ e PBIA entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa estende o da Seção 8.3, agora posicionando
os **cinco documentos nacionais/de bloco ao mesmo tempo** — PBIA (Brasil),
AI+ (China, 2023), China New Gen (China, 2017), Apply AI Strategy (UE, 2025)
e **America's AI Action Plan (EUA, 2025)** — no mesmo eixo BRICS × OCDE. Como
antes, cada documento aparece com dois pontos (cosseno, o principal, e
Jaccard, o secundário, ligados por uma linha pontilhada), e os cantos de
referência marcam onde o próprio BRICS e a própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

national_docs = [
    ("PBIA", CATEGORICAL_PBIA, jaccard_pbia_brics, cosine_pbia_brics, jaccard_pbia_ocde, cosine_pbia_ocde, (10, 10)),
    ("AI+", CATEGORICAL_AIPLUS, jaccard_aiplus_brics, cosine_aiplus_brics, jaccard_aiplus_ocde, cosine_aiplus_ocde, (10, -14)),
    ("China New Gen", CATEGORICAL_CHINA, jaccard_china_brics, cosine_china_brics, jaccard_china_ocde, cosine_china_ocde, (10, -14)),
    ("Apply AI Strategy", CATEGORICAL_APPLY, jaccard_apply_brics, cosine_apply_brics, jaccard_apply_ocde, cosine_apply_ocde, (10, 10)),
    (AMERICA_LABEL, CATEGORICAL_AMERICA, jaccard_america_brics, cosine_america_brics, jaccard_america_ocde, cosine_america_ocde, (-12, 12)),
]

for name, color, jac_x, cos_x, jac_y, cos_y, offset in national_docs:
    ax.plot([jac_x, cos_x], [jac_y, cos_y], linestyle=":", linewidth=1.0, color=color, zorder=4)
    ax.scatter([jac_x], [jac_y], s=110, color=color, alpha=0.45,
               edgecolors=SURFACE, linewidths=0.8, zorder=5, label=f"{name} (Jaccard)")
    ax.scatter([cos_x], [cos_y], s=340, color=color, alpha=0.92,
               edgecolors=SURFACE, linewidths=1.2, zorder=6, label=f"{name} (cosseno)")
    ha = "right" if offset[0] < 0 else "left"
    ax.annotate(name, (cos_x, cos_y), textcoords="offset points", xytext=offset,
                ha=ha, fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA, AI+, China New Gen, Apply AI Strategy e America's AI Action Plan no mapa BRICS × OCDE",
             color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8)

fig.tight_layout()
plt.show()

### Síntese — America's AI Action Plan, BRICS, OCDE, PBIA, AI+, China New Gen e Apply AI Strategy

- O America's AI Action Plan **também pende para o eixo do BRICS em vez do
  eixo da OCDE**, replicando o padrão dos quatro documentos nacionais/de
  bloco anteriores: cosseno de **0,447** com o BRICS contra **0,375** com a
  OCDE (margem de 0,072), e Jaccard de **0,168** contra **0,150**. É a
  **terceira menor margem** entre os cinco planos (0,143 no PBIA, 0,087 no
  AI+, 0,068 no China New Gen e no Apply AI Strategy, 0,072 no America's AI
  Action Plan) — o plano americano fica quase equilibrado entre os dois
  eixos, mas ainda claramente do lado do BRICS.
- O achado mais notável desta seção: a **maior similaridade absoluta do
  America's AI Action Plan não é com o BRICS**, e sim com o **China New Gen
  (cosseno 0,485)** — acima até do próprio BRICS. Em ordem decrescente:
  China New Gen (0,485), Apply AI Strategy (0,462), PBIA (0,457), BRICS
  (0,447), AI+ (0,423) e, por último, OCDE (0,375). Apesar da retórica
  explícita de rivalidade geopolítica com a China no próprio texto do
  documento, o vocabulário institucional dos dois planos — ambos extensos,
  organizados em pilares/ações concretas de política industrial e
  tecnológica (*systems*, *development*, *security*, *infrastructure*,
  *standards*) — os aproxima mais do que qualquer bloco político. Pelo
  Jaccard (sobreposição bruta, sem peso de frequência), a ordem muda
  ligeiramente: o Apply AI Strategy lidera (0,234), seguido do PBIA (0,209),
  China New Gen (0,181), AI+ (0,173), BRICS (0,168) e OCDE, novamente a
  menor (0,150).
- No **eco de vocabulário** (10.2), o termo *development* é o que mais pende
  para o BRICS entre os termos frequentes do America's AI Action Plan
  (diferença de +8,82 ocorrências por 1.000 tokens a favor do BRICS), com
  *critical*, *national* e *infrastructure* na mesma direção. Já *systems* é
  o termo que mais pende para a OCDE (-5,53), acompanhado de *new*,
  *adoption* e *research*. Uma parte importante do vocabulário mais
  frequente do plano americano — *federal*, *frontier*, *manufacturing*,
  *agencies* — simplesmente **não aparece nem no BRICS nem na OCDE**,
  refletindo o registro institucional-regulatório específico dos EUA
  (agências federais nomeadas uma a uma, "frontier AI models", manufatura
  doméstica de semicondutores) que não tem equivalente direto em nenhum dos
  dois documentos de referência.
- No **mapa combinado** (10.3), o America's AI Action Plan aparece deslocado
  dos outros quatro pontos nacionais, mais próximo da diagonal BRICS=OCDE do
  que PBIA, AI+, China New Gen ou Apply AI Strategy — consequência direta da
  margem pequena (0,072) entre suas duas similaridades. Ainda assim, seu
  ponto permanece do lado do eixo X (BRICS), fechando o padrão: **os cinco
  documentos nacionais/regionais do estudo pendem, todos, para o eixo do
  BRICS em vez do eixo da OCDE** — nenhum se aproxima mais da arquitetura de
  governança da OCDE do que da agenda de desenvolvimento do BRICS.

## 11. Síntese final — os sete documentos juntos

Estendemos o mapa de escalonamento multidimensional (MDS) da Seção 9 para
incluir o **America's AI Action Plan** como um sétimo documento, ao lado de
BRICS, OCDE, AI+, PBIA, China New Gen e Apply AI Strategy. Construímos a
matriz completa de distâncias entre os sete documentos (distância = 1 −
similaridade de cosseno) e reaproveitamos a função `classical_mds` já
definida na Seção 5.2, mantendo os dois componentes de maior variância.

In [ ]:
doc_labels7 = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL, CHINA_LABEL, APPLY_LABEL, AMERICA_LABEL]
doc_colors7 = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA, CATEGORICAL_CHINA, CATEGORICAL_APPLY, CATEGORICAL_AMERICA]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA, China New Gen, Apply AI Strategy, America's AI Action Plan)
S7 = np.array([
    [1.0,                 cosine_similarity,    cosine_aiplus_brics, cosine_pbia_brics, cosine_china_brics, cosine_apply_brics, cosine_america_brics],
    [cosine_similarity,   1.0,                  cosine_aiplus_ocde,  cosine_pbia_ocde,  cosine_china_ocde,  cosine_apply_ocde,  cosine_america_ocde],
    [cosine_aiplus_brics, cosine_aiplus_ocde,   1.0,                 cosine_aiplus_pbia, cosine_china_aiplus, cosine_apply_aiplus, cosine_america_aiplus],
    [cosine_pbia_brics,   cosine_pbia_ocde,     cosine_aiplus_pbia,  1.0,                cosine_china_pbia,  cosine_apply_pbia,  cosine_america_pbia],
    [cosine_china_brics,  cosine_china_ocde,    cosine_china_aiplus, cosine_china_pbia,  1.0,                cosine_apply_china,  cosine_america_china],
    [cosine_apply_brics,  cosine_apply_ocde,    cosine_apply_aiplus, cosine_apply_pbia,  cosine_apply_china, 1.0,                 cosine_america_apply],
    [cosine_america_brics, cosine_america_ocde, cosine_america_aiplus, cosine_america_pbia, cosine_america_china, cosine_america_apply, 1.0],
])
D7 = 1.0 - S7
np.fill_diagonal(D7, 0.0)

coords7, explained_variance7 = classical_mds(D7)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels7, coords7):
    print(f"  {label:24s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance7*100:.1f}%")

fig, ax = plt.subplots(figsize=(9, 9), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels7)):
    for j in range(i + 1, len(doc_labels7)):
        sim = S7[i, j]
        ax.plot([coords7[i, 0], coords7[j, 0]], [coords7[i, 1], coords7[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels7, doc_colors7, coords7):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords7).max() * 0.45, 0.05)
ax.set_xlim(coords7[:, 0].min() - pad, coords7[:, 0].max() + pad)
ax.set_ylim(coords7[:, 1].min() - pad, coords7[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos sete documentos (MDS clássico, {explained_variance7*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

### Síntese — o mapa completo dos sete documentos

- A variância explicada pelos dois componentes cai para **56,3%** (ante
  66,5% com seis documentos) — com sete documentos, o espaço de similaridade
  textual já não cabe bem em duas dimensões, então o mapa deve ser lido
  estritamente como um retrato aproximado da estrutura geral (agrupamentos),
  não como distâncias exatas par a par. As similaridades de cosseno
  calculadas nas seções anteriores continuam sendo a medida precisa.
- A ligação mais forte de todo o conjunto de sete documentos (21 pares)
  permanece, disparado, **China New Gen–AI+ (0,742)** — quase 0,2 ponto
  acima do segundo par mais próximo, **BRICS–PBIA (0,538)**, seguido de
  **BRICS–OCDE (0,519)**. A entrada do America's AI Action Plan não altera
  essa hierarquia: seu par mais forte, **China New Gen–America (0,485)**, só
  aparece na 5ª posição geral.
- O achado mais robusto do estudo completo se confirma e se fortalece: as
  **cinco menores similaridades de todo o conjunto de 21 pares são,
  exatamente, as cinco similaridades da OCDE com os cinco planos
  nacionais/regionais** — AI+ (0,330, a mais baixa de todas), Apply AI
  Strategy (0,375), America's AI Action Plan (0,375, empatada), China New
  Gen (0,379) e PBIA (0,395). A única conexão relevante da OCDE continua
  sendo o **BRICS** (0,519) — as duas únicas "declarações de princípios" do
  conjunto. No mapa, a OCDE aparece isolada no canto inferior esquerdo,
  visivelmente afastada de todos os demais pontos, incluindo agora o
  America's AI Action Plan.
- No mapa 2D, o America's AI Action Plan aparece posicionado **entre PBIA e
  Apply AI Strategy** — mais próximo desses dois pontos no plano do que do
  cluster China New Gen–AI+, ainda que sua maior similaridade de cosseno
  bruta seja com o China New Gen. Isso é um efeito esperado da projeção: com
  apenas 56,3% da variância capturada em duas dimensões, o MDS prioriza
  preservar a estrutura de distâncias do conjunto como um todo, e o
  America's AI Action Plan tem similaridades comparáveis com três documentos
  ao mesmo tempo (China New Gen 0,485, Apply AI Strategy 0,462, PBIA 0,457)
  — por isso seu ponto se acomoda numa posição intermediária, equidistante
  desse trio, em vez de colar em qualquer um dos dois clusters já
  estabelecidos.
- Em conjunto, o mapa de sete documentos consolida três agrupamentos nítidos
  observados desde a Seção 9: (1) as duas políticas chinesas (AI+ e China
  New Gen), o par mais próximo de todo o estudo por larga margem; (2) os
  planos de ação setoriais — PBIA, Apply AI Strategy e agora o America's AI
  Action Plan —, com o BRICS funcionando como elo próximo de todos eles; e
  (3) a OCDE, isolada como o único documento de governança pura do conjunto,
  cujas cinco piores similaridades no estudo inteiro são, sem exceção, com
  os cinco planos nacionais/regionais analisados.

## 12. AI Continent Action Plan (UE) — a qual documento a nova comunicação da Comissão se aproxima mais?

O **AI Continent Action Plan** (`AI_CONTINENT_ACTION_PLAN.json`) é a comunicação
da Comissão Europeia COM(2025) 165 final (abril de 2025) — o plano de
infraestrutura de IA da UE, estruturado em cinco eixos (computação, dados,
adoção setorial, competências e simplificação regulatória). Assim como o
BRICS, a OCDE, o PBIA, o AI+ e o Apply AI Strategy, seu JSON guarda o texto em
uma lista simples de seções com campo `text`, então a mesma função
`load_tokens` usada para esses documentos se aplica diretamente, sem
necessidade de um carregador próprio.

Comparamos o AI Continent Action Plan com os sete documentos já carregados —
BRICS Declaration, OECD Recommendation, PBIA, AI+, China New Gen, Apply AI
Strategy e America's AI Action Plan — para responder duas perguntas: (1)
sendo uma comunicação da própria Comissão Europeia, também autora do Apply AI
Strategy, o quanto os dois documentos europeus se parecem entre si?; e (2)
descontada essa proximidade institucional esperada, o AI Continent Action
Plan replica o padrão dos seis documentos nacionais/de bloco anteriores,
pendendo para o eixo **desenvolvimento/economia digital** do BRICS em vez do
eixo **arquitetura de governança** da OCDE?

In [ ]:
CONTINENT_LABEL = "AI Continent Action Plan (UE)"

continent_tokens, continent_data = load_tokens("AI_CONTINENT_ACTION_PLAN.json")
continent_counts = Counter(continent_tokens)

print(f"{CONTINENT_LABEL}: {len(continent_tokens)} tokens, {len(continent_counts)} termos únicos")


### 12.1. Similaridade do AI Continent Action Plan com cada documento

Reaproveitamos o índice de Jaccard e a similaridade de cosseno definidos na
Seção 1, agora aplicados aos pares (AI Continent Action Plan, BRICS), (AI
Continent Action Plan, OCDE), (AI Continent Action Plan, PBIA), (AI Continent
Action Plan, AI+), (AI Continent Action Plan, China New Gen), (AI Continent
Action Plan, Apply AI Strategy) e (AI Continent Action Plan, America's AI
Action Plan).

In [ ]:
jaccard_continent_brics, cosine_continent_brics = jaccard_cosine(continent_counts, continent_tokens, brics_counts, brics_tokens)
jaccard_continent_ocde, cosine_continent_ocde = jaccard_cosine(continent_counts, continent_tokens, ocde_counts, ocde_tokens)
jaccard_continent_pbia, cosine_continent_pbia = jaccard_cosine(continent_counts, continent_tokens, pbia_counts, pbia_tokens)
jaccard_continent_aiplus, cosine_continent_aiplus = jaccard_cosine(continent_counts, continent_tokens, aiplus_counts, aiplus_tokens)
jaccard_continent_china, cosine_continent_china = jaccard_cosine(continent_counts, continent_tokens, china_counts, china_tokens)
jaccard_continent_apply, cosine_continent_apply = jaccard_cosine(continent_counts, continent_tokens, apply_counts, apply_tokens)
jaccard_continent_america, cosine_continent_america = jaccard_cosine(continent_counts, continent_tokens, america_counts, america_tokens)

print(f"{CONTINENT_LABEL} × {BRICS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_continent_brics:.3f}")
print(f"  Similaridade de cosseno: {cosine_continent_brics:.3f}")
print()
print(f"{CONTINENT_LABEL} × {OECD_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_continent_ocde:.3f}")
print(f"  Similaridade de cosseno: {cosine_continent_ocde:.3f}")
print()
print(f"{CONTINENT_LABEL} × {PBIA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_continent_pbia:.3f}")
print(f"  Similaridade de cosseno: {cosine_continent_pbia:.3f}")
print()
print(f"{CONTINENT_LABEL} × {AIPLUS_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_continent_aiplus:.3f}")
print(f"  Similaridade de cosseno: {cosine_continent_aiplus:.3f}")
print()
print(f"{CONTINENT_LABEL} × {CHINA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_continent_china:.3f}")
print(f"  Similaridade de cosseno: {cosine_continent_china:.3f}")
print()
print(f"{CONTINENT_LABEL} × {APPLY_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_continent_apply:.3f}")
print(f"  Similaridade de cosseno: {cosine_continent_apply:.3f}")
print()
print(f"{CONTINENT_LABEL} × {AMERICA_LABEL}:")
print(f"  Índice de Jaccard:       {jaccard_continent_america:.3f}")
print(f"  Similaridade de cosseno: {cosine_continent_america:.3f}")


### Gráfico — com qual documento o AI Continent Action Plan se parece mais?

As sete comparações lado a lado (AI Continent Action Plan × BRICS, × OCDE, ×
PBIA, × AI+, × China New Gen, × Apply AI Strategy, × America's AI Action
Plan), para as duas métricas.

In [ ]:
CATEGORICAL_CONTINENT = "#1a3d7c"  # cor de identidade do AI Continent Action Plan nos gráficos desta seção

metric_labels = ["Índice de Jaccard\n(vocabulário)", "Similaridade de cosseno\n(ênfase)"]
brics_values = [jaccard_continent_brics, cosine_continent_brics]
ocde_values = [jaccard_continent_ocde, cosine_continent_ocde]
pbia_values = [jaccard_continent_pbia, cosine_continent_pbia]
aiplus_values = [jaccard_continent_aiplus, cosine_continent_aiplus]
china_values = [jaccard_continent_china, cosine_continent_china]
apply_values = [jaccard_continent_apply, cosine_continent_apply]
america_values = [jaccard_continent_america, cosine_continent_america]

fig, ax = plt.subplots(figsize=(8, 7.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = list(range(len(metric_labels)))
bar_h = 0.11
offsets = [3 * bar_h, 2 * bar_h, 1 * bar_h, 0.0, -1 * bar_h, -2 * bar_h, -3 * bar_h]

series = [
    (brics_values, offsets[0], CATEGORICAL_BLUE, f"{CONTINENT_LABEL} × {BRICS_LABEL}"),
    (ocde_values, offsets[1], CATEGORICAL_AQUA, f"{CONTINENT_LABEL} × {OECD_LABEL}"),
    (pbia_values, offsets[2], CATEGORICAL_PBIA, f"{CONTINENT_LABEL} × {PBIA_LABEL}"),
    (aiplus_values, offsets[3], CATEGORICAL_AIPLUS, f"{CONTINENT_LABEL} × {AIPLUS_LABEL}"),
    (china_values, offsets[4], CATEGORICAL_CHINA, f"{CONTINENT_LABEL} × {CHINA_LABEL}"),
    (apply_values, offsets[5], CATEGORICAL_APPLY, f"{CONTINENT_LABEL} × {APPLY_LABEL}"),
    (america_values, offsets[6], CATEGORICAL_AMERICA, f"{CONTINENT_LABEL} × {AMERICA_LABEL}"),
]

for values, offset, color, label in series:
    ys = [y + offset for y in y_pos]
    ax.barh(ys, values, height=bar_h, color=color, zorder=3, label=label)
    for y, v in zip(ys, values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=8.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade (0 a 1)", color=INK_MUTED, fontsize=10)
all_values = brics_values + ocde_values + pbia_values + aiplus_values + china_values + apply_values + america_values
ax.set_xlim(0, max(all_values) * 1.25)
ax.set_title("Com qual documento o AI Continent Action Plan se parece mais?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout()
plt.show()


### 12.2. O vocabulário mais usado pelo AI Continent Action Plan ecoa mais no BRICS ou na OCDE?

Para os termos mais frequentes do AI Continent Action Plan (excluindo
palavras autorreferentes como *european*, *europe*, *commission* e *eu* — as
mesmas excluídas para o Apply AI Strategy, já que ambos são comunicações da
Comissão Europeia), comparamos o quanto **BRICS** e **OCDE** usam cada um
desses mesmos termos — a mesma análise feita para os seis documentos
anteriores.

In [ ]:
SELF_REFERENTIAL_CONTINENT = {"european", "europe", "commission", "eu"}
MIN_CONTINENT_COUNT = 3
TOP_N_CONTINENT_TERMS = 18

continent_terms_rows = []
for t, continent_c in continent_counts.most_common():
    if t in SELF_REFERENTIAL_CONTINENT or continent_c < MIN_CONTINENT_COUNT:
        continue
    b = relative_freq(brics_counts, len(brics_tokens), t)
    o = relative_freq(ocde_counts, len(ocde_tokens), t)
    continent_terms_rows.append((t, b, o, b - o))
    if len(continent_terms_rows) >= TOP_N_CONTINENT_TERMS:
        break

continent_terms_rows.sort(key=lambda x: x[3])  # OCDE-leaning no topo, BRICS-leaning na base

labels = [r[0] for r in continent_terms_rows]
brics_vals = [r[1] for r in continent_terms_rows]
ocde_vals = [r[2] for r in continent_terms_rows]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(continent_terms_rows) + 1), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

y_pos = range(len(continent_terms_rows))
ax.barh(y_pos, [-v for v in brics_vals], color=CATEGORICAL_BLUE, height=0.6, zorder=3, label=BRICS_LABEL)
ax.barh(y_pos, ocde_vals, color=CATEGORICAL_AQUA, height=0.6, zorder=3, label=OECD_LABEL)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9)
ax.axvline(0, color=GRIDLINE, linewidth=1.0, zorder=2)

ax.set_xlabel("Frequência relativa em cada documento (ocorrências por 1.000 tokens)", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Vocabulário mais usado pelo AI Continent Action Plan: quem mais o ecoa — {BRICS_LABEL} (esquerda) × {OECD_LABEL} (direita)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", pad=14,
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)

max_abs = max(max(brics_vals), max(ocde_vals), 1e-6)
ax.set_xlim(-max_abs * 1.2, max_abs * 1.2)
xticks = ax.get_xticks()
ax.set_xticks(xticks)
ax.set_xticklabels([f"{abs(x):.0f}" for x in xticks], color=INK_MUTED, fontsize=9)

ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=9)

fig.tight_layout()
plt.show()


### 12.3. Mapa combinado — AI Continent Action Plan junto a America's AI Action Plan, China New Gen, Apply AI Strategy, AI+ e PBIA entre BRICS e OCDE

Eixo X: similaridade com o **BRICS Declaration**. Eixo Y: similaridade com a
**OECD Recommendation**. Este mapa estende o da Seção 10.3, agora
posicionando os **seis documentos nacionais/de bloco e supranacionais ao
mesmo tempo** — PBIA (Brasil), AI+ (China, 2023), China New Gen (China,
2017), Apply AI Strategy (UE, 2025), America's AI Action Plan (EUA, 2025) e
**AI Continent Action Plan (UE, 2025)** — no mesmo eixo BRICS × OCDE. Como
antes, cada documento aparece com dois pontos (cosseno, o principal, e
Jaccard, o secundário, ligados por uma linha pontilhada), e os cantos de
referência marcam onde o próprio BRICS e a própria OCDE cairiam neste mapa.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 9.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

axis_max = 1.05
ax.plot([0, axis_max], [0, axis_max], linestyle="--", linewidth=1.2, color=INK_MUTED, zorder=2)

# Pontos de referência: onde o próprio BRICS e a própria OCDE cairiam neste mapa
ax.scatter([1.0], [cosine_similarity], s=140, color=CATEGORICAL_BLUE, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{BRICS_LABEL} (referência)")
ax.annotate(BRICS_LABEL, (1.0, cosine_similarity), textcoords="offset points", xytext=(-8, 8),
            ha="right", fontsize=8.5, color=INK_MUTED)

ax.scatter([cosine_similarity], [1.0], s=140, color=CATEGORICAL_AQUA, alpha=0.35,
           edgecolors=SURFACE, linewidths=0.8, zorder=3, label=f"{OECD_LABEL} (referência)")
ax.annotate(OECD_LABEL, (cosine_similarity, 1.0), textcoords="offset points", xytext=(8, -10),
            ha="left", fontsize=8.5, color=INK_MUTED)

national_docs = [
    ("PBIA", CATEGORICAL_PBIA, jaccard_pbia_brics, cosine_pbia_brics, jaccard_pbia_ocde, cosine_pbia_ocde, (10, 10)),
    ("AI+", CATEGORICAL_AIPLUS, jaccard_aiplus_brics, cosine_aiplus_brics, jaccard_aiplus_ocde, cosine_aiplus_ocde, (10, -14)),
    ("China New Gen", CATEGORICAL_CHINA, jaccard_china_brics, cosine_china_brics, jaccard_china_ocde, cosine_china_ocde, (10, -14)),
    ("Apply AI Strategy", CATEGORICAL_APPLY, jaccard_apply_brics, cosine_apply_brics, jaccard_apply_ocde, cosine_apply_ocde, (10, 10)),
    (AMERICA_LABEL, CATEGORICAL_AMERICA, jaccard_america_brics, cosine_america_brics, jaccard_america_ocde, cosine_america_ocde, (-12, 12)),
    (CONTINENT_LABEL, CATEGORICAL_CONTINENT, jaccard_continent_brics, cosine_continent_brics, jaccard_continent_ocde, cosine_continent_ocde, (12, -16)),
]

for name, color, jac_x, cos_x, jac_y, cos_y, offset in national_docs:
    ax.plot([jac_x, cos_x], [jac_y, cos_y], linestyle=":", linewidth=1.0, color=color, zorder=4)
    ax.scatter([jac_x], [jac_y], s=110, color=color, alpha=0.45,
               edgecolors=SURFACE, linewidths=0.8, zorder=5, label=f"{name} (Jaccard)")
    ax.scatter([cos_x], [cos_y], s=340, color=color, alpha=0.92,
               edgecolors=SURFACE, linewidths=1.2, zorder=6, label=f"{name} (cosseno)")
    ha = "right" if offset[0] < 0 else "left"
    ax.annotate(name, (cos_x, cos_y), textcoords="offset points", xytext=offset,
                ha=ha, fontsize=11, fontweight="bold", color=INK_PRIMARY)

ax.set_xlabel(f"Similaridade com o {BRICS_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_ylabel(f"Similaridade com a {OECD_LABEL} (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_title("PBIA, AI+, China New Gen, Apply AI Strategy, America's AI Action Plan e AI Continent Action Plan no mapa BRICS × OCDE",
             color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=14)
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8)

fig.tight_layout()
plt.show()


### Síntese — AI Continent Action Plan, BRICS, OCDE, PBIA, AI+, China New Gen, Apply AI Strategy e America's AI Action Plan

- A similaridade mais forte do AI Continent Action Plan — e, disparada, **a
  maior similaridade de todo o conjunto de documentos analisados até aqui** —
  é com o **Apply AI Strategy (cosseno 0,773; Jaccard 0,362)**, a outra
  comunicação da Comissão Europeia do estudo. O par supera até o antigo
  recorde **AI+–China New Gen (0,742)**, as duas políticas chinesas, por 3,1
  pontos percentuais. Faz sentido institucionalmente: os dois textos
  compartilham autor (Comissão Europeia), ano (2025), e uma rede densa de
  iniciativas citadas nos dois lados — AI Factories, RAISE, AI Skills
  Academy, Data Union Strategy —, já que o próprio AI Continent Action Plan é
  o documento que **anuncia** o Apply AI Strategy como comunicação futura.
- Descontado esse par institucional, o AI Continent Action Plan **replica o
  padrão dos seis documentos nacionais/de bloco anteriores**: pende para o
  eixo do **BRICS (cosseno 0,453)** em vez do eixo da **OCDE (cosseno
  0,368)**, com margem de **0,085** — a **segunda maior margem** entre os
  seis planos analisados até aqui, atrás apenas do PBIA (0,143) e
  praticamente empatada com o AI+ (0,087); à frente da America's AI Action
  Plan (0,072), do China New Gen e do Apply AI Strategy (0,068 nos dois). A
  OCDE volta a ser, como em todos os casos anteriores, o documento com que
  o texto analisado menos se parece.
- Em ordem decrescente de similaridade de cosseno com o AI Continent Action
  Plan: Apply AI Strategy (0,773), PBIA (0,476), America's AI Action Plan
  (0,465), BRICS (0,453), China New Gen (0,408), AI+ (0,369) e, por último,
  OCDE (0,368) — AI+ e OCDE ficam praticamente empatados na última posição,
  a uma diferença de apenas 0,001.
- No **eco de vocabulário** (12.2), os termos mais frequentes do AI Continent
  Action Plan pendem fortemente para o BRICS: *digital* (+9,50 ocorrências
  por 1.000 tokens a favor do BRICS), *development* (+8,82) e *innovation*
  (+6,37) lideram essa direção. Do lado oposto, apenas *research* pende para
  a OCDE (-1,42), com margem pequena. Quatro termos centrais do documento —
  *strategy*, *factories*, *member* e *computing* — **não aparecem nem no
  BRICS nem na OCDE**, o vocabulário de infraestrutura específico da UE (AI
  Factories, Estados-Membros, capacidade computacional) sem equivalente
  direto em nenhum dos dois documentos de referência — o mesmo fenômeno
  observado com o jargão federal-regulatório do America's AI Action Plan.
- No **mapa combinado** (12.3), o AI Continent Action Plan aparece no mesmo
  quadrante dos demais cinco documentos nacionais/de bloco, do lado do eixo
  BRICS da diagonal — mas este mapa, por construção, só enxerga a
  similaridade de cada documento com BRICS e com OCDE; a proximidade extrema
  com o Apply AI Strategy (0,773) não aparece aqui, já que os dois pontos têm
  coordenadas BRICS/OCDE parecidas, mas não idênticas. Essa proximidade só
  fica visível no mapa de síntese completo (Seção 13), que considera todas as
  distâncias par a par de uma vez.

## 13. Síntese final — os oito documentos juntos

Estendemos o mapa de escalonamento multidimensional (MDS) da Seção 11 para
incluir o **AI Continent Action Plan** como um oitavo documento, ao lado de
BRICS, OCDE, AI+, PBIA, China New Gen, Apply AI Strategy e America's AI
Action Plan. Construímos a matriz completa de distâncias entre os oito
documentos (distância = 1 − similaridade de cosseno) e reaproveitamos a
função `classical_mds` já definida na Seção 5.2, mantendo os dois componentes
de maior variância.

In [ ]:
doc_labels8 = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL, CHINA_LABEL, APPLY_LABEL, AMERICA_LABEL, CONTINENT_LABEL]
doc_colors8 = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA, CATEGORICAL_CHINA, CATEGORICAL_APPLY, CATEGORICAL_AMERICA, CATEGORICAL_CONTINENT]

# Matriz de similaridade de cosseno (ordem: BRICS, OCDE, AI+, PBIA, China New Gen, Apply AI Strategy, America's AI Action Plan, AI Continent Action Plan)
S8 = np.array([
    [1.0,                  cosine_similarity,     cosine_aiplus_brics,   cosine_pbia_brics,   cosine_china_brics,   cosine_apply_brics,   cosine_america_brics,   cosine_continent_brics],
    [cosine_similarity,    1.0,                   cosine_aiplus_ocde,    cosine_pbia_ocde,    cosine_china_ocde,    cosine_apply_ocde,    cosine_america_ocde,    cosine_continent_ocde],
    [cosine_aiplus_brics,  cosine_aiplus_ocde,    1.0,                   cosine_aiplus_pbia,  cosine_china_aiplus,  cosine_apply_aiplus,  cosine_america_aiplus,  cosine_continent_aiplus],
    [cosine_pbia_brics,    cosine_pbia_ocde,      cosine_aiplus_pbia,    1.0,                 cosine_china_pbia,    cosine_apply_pbia,    cosine_america_pbia,    cosine_continent_pbia],
    [cosine_china_brics,   cosine_china_ocde,     cosine_china_aiplus,   cosine_china_pbia,   1.0,                  cosine_apply_china,   cosine_america_china,   cosine_continent_china],
    [cosine_apply_brics,   cosine_apply_ocde,     cosine_apply_aiplus,   cosine_apply_pbia,   cosine_apply_china,   1.0,                  cosine_america_apply,   cosine_continent_apply],
    [cosine_america_brics, cosine_america_ocde,   cosine_america_aiplus, cosine_america_pbia, cosine_america_china, cosine_america_apply, 1.0,                    cosine_continent_america],
    [cosine_continent_brics, cosine_continent_ocde, cosine_continent_aiplus, cosine_continent_pbia, cosine_continent_china, cosine_continent_apply, cosine_continent_america, 1.0],
])
D8 = 1.0 - S8
np.fill_diagonal(D8, 0.0)

coords8, explained_variance8 = classical_mds(D8)
print("Coordenadas MDS (dim 1, dim 2):")
for label, (x, y) in zip(doc_labels8, coords8):
    print(f"  {label:28s} ({x:+.3f}, {y:+.3f})")
print(f"\nVariância explicada pelos 2 componentes: {explained_variance8*100:.1f}%")

fig, ax = plt.subplots(figsize=(9.5, 9.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

# Linhas conectando todos os pares, com espessura proporcional à similaridade
# (par mais similar = linha mais grossa e mais opaca)
for i in range(len(doc_labels8)):
    for j in range(i + 1, len(doc_labels8)):
        sim = S8[i, j]
        ax.plot([coords8[i, 0], coords8[j, 0]], [coords8[i, 1], coords8[j, 1]],
                color=INK_MUTED, linewidth=0.6 + sim * 3.2, alpha=0.18 + sim * 0.35, zorder=1)

for label, color, (x, y) in zip(doc_labels8, doc_colors8, coords8):
    ax.scatter([x], [y], s=420, color=color, alpha=0.9, edgecolors=SURFACE,
               linewidths=1.6, zorder=3)
    ax.annotate(label, (x, y), textcoords="offset points", xytext=(0, 20),
                ha="center", fontsize=11, fontweight="bold", color=INK_PRIMARY, zorder=4)

ax.axhline(0, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.axvline(0, color=GRIDLINE, linewidth=0.8, zorder=0)

pad = max(np.abs(coords8).max() * 0.45, 0.05)
ax.set_xlim(coords8[:, 0].min() - pad, coords8[:, 0].max() + pad)
ax.set_ylim(coords8[:, 1].min() - pad, coords8[:, 1].max() + pad)

ax.set_xlabel("Dimensão MDS 1", color=INK_MUTED, fontsize=10)
ax.set_ylabel("Dimensão MDS 2", color=INK_MUTED, fontsize=10)
ax.set_title(
    f"Mapa de similaridade textual dos oito documentos (MDS clássico, {explained_variance8*100:.0f}% da variância)",
    color=INK_PRIMARY, fontsize=12.5, fontweight="bold", pad=16,
)

for spine in ax.spines.values():
    spine.set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### Síntese — o mapa completo dos oito documentos

- A variância explicada pelos dois componentes sobe ligeiramente para
  **57,6%** (ante 56,3% com sete documentos) — ainda uma fração modesta, então
  o mapa 2D continua sendo uma leitura aproximada da estrutura de
  agrupamentos, não das distâncias exatas par a par (essas seguem nas
  similaridades de cosseno calculadas seção a seção).
- A ligação mais forte de todo o conjunto de oito documentos (28 pares) passa
  a ser, com folga, **Apply AI Strategy–AI Continent Action Plan (0,773)** —
  a maior similaridade de todo o estudo, superando o antigo recorde
  **AI+–China New Gen (0,742)**. No mapa, os dois pontos aparecem colados um
  ao outro, visivelmente mais próximos entre si do que qualquer outro par de
  pontos do gráfico — o reflexo geométrico direto de serem as duas
  comunicações da mesma Comissão Europeia, publicadas no mesmo ano.
- Confirma-se e se aprofunda o achado mais robusto do estudo: as **cinco
  menores similaridades de todo o conjunto de 28 pares continuam sendo as
  cinco similaridades da OCDE com os cinco planos nacionais/regionais
  originais** — AI+ (0,330, a mais baixa de todas), Apply AI Strategy e
  America's AI Action Plan (0,375, empatados), China New Gen (0,379) e PBIA
  (0,395) — e o AI Continent Action Plan se junta a esse grupo por baixo,
  com OCDE×Continent (0,368) ficando entre AI+ e os demais como a **segunda
  pior similaridade de todo o conjunto**. A única ligação relevante da OCDE
  continua sendo o **BRICS** (0,519). No mapa, a OCDE permanece isolada no
  canto inferior, visivelmente afastada de todos os demais pontos.
- Em ordem decrescente de similaridade com o AI Continent Action Plan:
  Apply AI Strategy (0,773) muito à frente, seguido de PBIA (0,476),
  America's AI Action Plan (0,465), BRICS (0,453), China New Gen (0,408),
  AI+ (0,369) e OCDE (0,368) — reafirmando que, tirando o par institucional
  europeu, o AI Continent Action Plan pende para o eixo do BRICS como todos
  os demais documentos nacionais/de bloco do estudo.
- Em conjunto, o mapa de oito documentos consolida **quatro** agrupamentos:
  (1) o par europeu **Apply AI Strategy–AI Continent Action Plan**, agora a
  dupla mais coesa de todo o estudo; (2) as duas políticas chinesas (**AI+ e
  China New Gen**), a segunda dupla mais coesa; (3) os planos de ação
  nacionais restantes — **PBIA** e **America's AI Action Plan** —, com o
  **BRICS** funcionando como elo próximo de praticamente todos os seis
  documentos nacionais/de bloco; e (4) a **OCDE**, isolada como o único
  documento de governança pura do conjunto, com as piores similaridades de
  todo o estudo com os seis planos nacionais/de bloco/supranacionais
  analisados.

## 14. Mapa de quadrantes — UE × China × BRICS × EUA, com o PBIA sobreposto

Reaproveitamos a mesma lógica da dispersão da Seção 2 (cada ponto é um termo,
posicionado pela frequência relativa em que aparece nos documentos), mas
agora com quatro referências geopolíticas dispostas em cruz, em vez de duas:

- **Eixo X negativo** — UE: combinação do *AI Continent Action Plan* com o
  *Apply AI Strategy*;
- **Eixo X positivo** — China: combinação do *China New Gen* com o *AI+*;
- **Eixo Y positivo** — *BRICS AI Governance Declaration*;
- **Eixo Y negativo** — *America's AI Action Plan* (EUA).

Para cada termo, a posição no eixo X é a diferença entre a frequência
relativa (por 1.000 tokens) no grupo China e no grupo UE; no eixo Y, a
diferença entre a frequência relativa no BRICS e no America's AI Action
Plan. Termos "puramente" ligados a um bloco ficam grudados no
respectivo eixo/canto; termos equilibrados entre os quatro (razão entre o
segundo e o primeiro colocado ≥ 0.75) aparecem em amarelo, como
**convergentes**.

Em seguida, sobrepomos (losangos vermelhos) os termos mais usados pelo PBIA
que também aparecem em pelo menos um dos quatro documentos de referência —
calculados com a mesma fórmula de posição. Isso mostra, para cada termo do
vocabulário brasileiro, a qual bloco geopolítico ele mais se aproxima.


In [ ]:
CATEGORICAL_UE_Q = CATEGORICAL_APPLY          # laranja — UE (Apply AI Strategy + AI Continent Action Plan)
CATEGORICAL_CHINA_Q = CATEGORICAL_CHINA        # rosa — China (China New Gen + AI+)
CATEGORICAL_BRICS_Q = CATEGORICAL_BLUE         # azul — BRICS Declaration
CATEGORICAL_AMERICA_Q = CATEGORICAL_AMERICA    # verde — America's AI Action Plan (EUA)
CATEGORICAL_CONVERGENT_Q = INK_MUTED           # cinza — convergente (sem bloco dominante), evita confundir com o laranja da UE

MIN_COMBINED_COUNT_QUAD = 6
BALANCE_RATIO_THRESHOLD_QUAD = 0.75

UE_Q_LABEL = "UE (AI Continent Action Plan + Apply AI Strategy)"
CHINA_Q_LABEL = "China (China New Gen + AI+)"

# Combina os dois documentos de cada bloco em um único corpus (contagens + total de tokens)
eu_group_counts = apply_counts + continent_counts
eu_group_total = len(apply_tokens) + len(continent_tokens)

china_group_counts = aiplus_counts + china_counts
china_group_total = len(aiplus_tokens) + len(china_tokens)

# Termos autorreferentes de cada bloco (já usados nas seções anteriores) não entram no mapa
SELF_REFERENTIAL_QUAD = (
    SELF_REFERENTIAL_AIPLUS | SELF_REFERENTIAL_CHINA | SELF_REFERENTIAL_APPLY
    | SELF_REFERENTIAL_CONTINENT | SELF_REFERENTIAL_AMERICA | {"brics"}
)

vocab_quad = (
    set(brics_counts) | set(america_counts) | set(eu_group_counts) | set(china_group_counts)
) - SELF_REFERENTIAL_QUAD

def quad_position(t):
    rel_brics = relative_freq(brics_counts, len(brics_tokens), t)
    rel_america = relative_freq(america_counts, len(america_tokens), t)
    rel_eu = relative_freq(eu_group_counts, eu_group_total, t)
    rel_china = relative_freq(china_group_counts, china_group_total, t)
    x = rel_china - rel_eu
    y = rel_brics - rel_america
    return x, y, rel_brics, rel_america, rel_eu, rel_china

def classify_quad(rel_brics, rel_america, rel_eu, rel_china):
    values = {"BRICS": rel_brics, "EUA": rel_america, "UE": rel_eu, "China": rel_china}
    ranked = sorted(values.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_val = ranked[0]
    ratio = ranked[1][1] / top_val
    return "convergente" if ratio >= BALANCE_RATIO_THRESHOLD_QUAD else top_label

quad_rows = []
for t in vocab_quad:
    combined_count = (
        brics_counts.get(t, 0) + america_counts.get(t, 0)
        + eu_group_counts.get(t, 0) + china_group_counts.get(t, 0)
    )
    if combined_count < MIN_COMBINED_COUNT_QUAD:
        continue
    x, y, rel_brics, rel_america, rel_eu, rel_china = quad_position(t)
    group = classify_quad(rel_brics, rel_america, rel_eu, rel_china)
    quad_rows.append((t, x, y, group))

color_map_quad = {
    "UE": CATEGORICAL_UE_Q, "China": CATEGORICAL_CHINA_Q,
    "BRICS": CATEGORICAL_BRICS_Q, "EUA": CATEGORICAL_AMERICA_Q,
    "convergente": CATEGORICAL_CONVERGENT_Q,
}
label_map_quad = {
    "UE": UE_Q_LABEL, "China": CHINA_Q_LABEL,
    "BRICS": BRICS_LABEL, "EUA": AMERICA_LABEL,
    "convergente": "Convergente (sem bloco dominante)",
}

groups_quad = {k: [] for k in color_map_quad}
for t, x, y, group in quad_rows:
    groups_quad[group].append((t, x, y))

fig, ax = plt.subplots(figsize=(9.5, 9.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

max_abs_quad = max(max(abs(x), abs(y)) for _, x, y, _ in quad_rows) * 1.15

ax.axvline(0, color=GRIDLINE, linewidth=1.2, zorder=2)
ax.axhline(0, color=GRIDLINE, linewidth=1.2, zorder=2)

for group_name in ["convergente", "UE", "China", "BRICS", "EUA"]:
    pts = groups_quad[group_name]
    if not pts:
        continue
    xs = [p[1] for p in pts]
    ys = [p[2] for p in pts]
    ax.scatter(xs, ys, s=42, color=color_map_quad[group_name], alpha=0.6, zorder=3,
               edgecolors=SURFACE, linewidths=0.5, label=label_map_quad[group_name])

# Rotula os termos mais extremos de cada bloco, para dar contexto ao gráfico
def top_extreme_quad(pts, key, n=3):
    return sorted(pts, key=key, reverse=True)[:n]

annotate_quad = (
    top_extreme_quad(groups_quad["China"], key=lambda p: p[1])
    + top_extreme_quad(groups_quad["UE"], key=lambda p: -p[1])
    + top_extreme_quad(groups_quad["BRICS"], key=lambda p: p[2])
    + top_extreme_quad(groups_quad["EUA"], key=lambda p: -p[2])
)
for t, x, y in annotate_quad:
    ax.annotate(t, (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8, color=INK_PRIMARY)

# --- PBIA sobreposto ---
MIN_PBIA_COUNT_QUAD = 3
MIN_REF_COUNT_FOR_PBIA = 2
TOP_N_PBIA_QUAD = 22

pbia_quad_candidates = []
for t, pbia_c in pbia_counts.items():
    if t in SELF_REFERENTIAL or t in SELF_REFERENTIAL_QUAD or pbia_c < MIN_PBIA_COUNT_QUAD:
        continue
    ref_count = (
        brics_counts.get(t, 0) + america_counts.get(t, 0)
        + eu_group_counts.get(t, 0) + china_group_counts.get(t, 0)
    )
    if ref_count < MIN_REF_COUNT_FOR_PBIA:
        continue
    x, y, *_ = quad_position(t)
    pbia_quad_candidates.append((t, x, y, pbia_c))

pbia_quad_candidates.sort(key=lambda r: r[3], reverse=True)
pbia_quad_points = pbia_quad_candidates[:TOP_N_PBIA_QUAD]

xs_pbia = [p[1] for p in pbia_quad_points]
ys_pbia = [p[2] for p in pbia_quad_points]
ax.scatter(xs_pbia, ys_pbia, s=95, color=CATEGORICAL_PBIA, alpha=0.95, zorder=5,
           marker="D", edgecolors=INK_PRIMARY, linewidths=0.7, label=PBIA_LABEL)
for t, x, y, _ in pbia_quad_points:
    ax.annotate(t, (x, y), textcoords="offset points", xytext=(6, -3), fontsize=8,
                color=INK_PRIMARY, fontweight="bold")

max_abs_quad = max(
    max_abs_quad,
    max([abs(x) for x in xs_pbia] + [0]) * 1.1,
    max([abs(y) for y in ys_pbia] + [0]) * 1.1,
)

ax.set_xlim(-max_abs_quad, max_abs_quad)
ax.set_ylim(-max_abs_quad, max_abs_quad)

ax.set_xlabel(
    "Diferença de frequência relativa — China menos UE (ocorrências por 1.000 tokens)",
    color=INK_MUTED, fontsize=10,
)
ax.set_ylabel(
    "Diferença de frequência relativa — BRICS menos EUA (ocorrências por 1.000 tokens)",
    color=INK_MUTED, fontsize=10,
)
ax.set_title(
    "Mapa de quadrantes — UE × China × BRICS × EUA, com o PBIA sobreposto",
    color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14,
)

# Textos de direção deslocados para fora da faixa y≈0 / x≈0, onde os termos mais
# extremos de cada bloco já ficam rotulados, para não colidir com eles
ax.text(0, max_abs_quad * 0.98, f"↑ {BRICS_LABEL}", ha="center", va="top",
        color=CATEGORICAL_BRICS_Q, fontsize=9, fontweight="bold")
ax.text(0, -max_abs_quad * 0.98, f"↓ {AMERICA_LABEL}", ha="center", va="bottom",
        color=CATEGORICAL_AMERICA_Q, fontsize=9, fontweight="bold")
ax.text(max_abs_quad * 0.98, max_abs_quad * 0.42, f"{CHINA_Q_LABEL} →", ha="right", va="center",
        color=CATEGORICAL_CHINA_Q, fontsize=9, fontweight="bold")
ax.text(-max_abs_quad * 0.98, max_abs_quad * 0.42, f"← {UE_Q_LABEL}", ha="left", va="center",
        color=CATEGORICAL_UE_Q, fontsize=9, fontweight="bold")

for spine in ["top", "right", "left", "bottom"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.grid(True, color=GRIDLINE, linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

ax.legend(loc="upper left", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5, bbox_to_anchor=(-0.02, 1.0))

fig.tight_layout()
plt.show()


### Síntese — o mapa de quadrantes

- Entre os 22 termos mais usados pelo PBIA que também aparecem em pelo menos
  um dos quatro documentos de referência, a divisão pelo eixo Y é nítida:
  **14 dos 22 termos (64%) penderam para o quadrante do BRICS**, contra
  apenas 8 para o *America's AI Action Plan* — confirmando, agora também no
  vocabulário específico do PBIA, o padrão de maior proximidade com o BRICS
  já visto nas seções anteriores. No eixo X a divisão é perfeitamente
  equilibrada: **11 termos penderam para a China e 11 para a UE**.
- O termo mais usado pelo PBIA entre os que aparecem nos quatro documentos,
  **"development"** (39 ocorrências), é também o ponto mais extremo do
  gráfico — grudado no canto **China + BRICS** — ecoando a retórica
  desenvolvimentista comum ao BRICS Declaration e ao par chinês (China New
  Gen / AI+). No mesmo canto ficam **"intelligence"**, **"technological"**,
  **"innovation"**, **"technologies"**, **"economic"** e **"global"**: o
  núcleo do vocabulário de inovação tecnológica do PBIA se aproxima mais do
  registro China+BRICS do que de qualquer outro bloco.
- Um pequeno grupo de termos revela uma aproximação inesperada com os EUA:
  **"new"**, **"national"** e **"plan"** penderam para o eixo do *America's
  AI Action Plan* (e, no caso de "new"/"national", também para o lado da
  China) — sinal de que o vocabulário de planejamento executivo do plano
  americano (o "plano nacional" de Trump) também ecoa no PBIA, apesar de o
  documento dos EUA divergir do bloco BRICS/China em quase tudo o mais.
- Do lado da UE, termos como **"data"**, **"sectors"**, **"public"** e
  **"potential"** aparecem no quadrante europeu, sinalizando que o
  vocabulário setorial e de dados do PBIA tem uma leitura mais próxima da
  moldura regulatória europeia (Apply AI Strategy + AI Continent Action
  Plan) do que da chinesa nesses pontos específicos.
- Em conjunto, o mapa mostra que o PBIA não se alinha a um único bloco: ele
  combina o vocabulário desenvolvimentista compartilhado por China e BRICS
  com fragmentos do registro regulatório europeu e, pontualmente, do
  vocabulário de execução do plano americano — um retrato mais granular do
  mesmo posicionamento "entre blocos" já sugerido pelo radar e pelo MDS das
  Seções 5, 7, 9 e 11.


## 15. O PBIA no centro — indicadores estruturais dos oito documentos

As seções 1 a 14 compararam os oito documentos pelo **vocabulário** (frequência
de termos, Jaccard, cosseno, MDS). Esta seção final muda de eixo: em vez de
*como cada documento fala*, olha para *o que cada documento efetivamente
declara ter* — usando diretamente os campos quantitativos e estruturais já
presentes em cada JSON (eixos/pilares, ações concretas, investimento, páginas,
volume textual e horizonte temporal), sem recorrer a frequência de palavras.
O PBIA é o eixo de leitura: cada gráfico mostra os outros sete documentos
**em relação ao PBIA**, não em ranking absoluto.

**Fonte de cada indicador (nenhum valor é inferido por frequência de texto —
todos vêm de um campo explícito do JSON correspondente):**

| Documento | Páginas | Blocos estruturantes | Ações concretas | Investimento | Horizonte |
|---|---|---|---|---|---|
| PBIA | `document.pages` | `structuring_axes` (5 eixos) | `immediate_impact_actions` + `structuring_actions` (27+54) | `investments.total_brl_billion` | fim de `investments.period` (2024–2028) |
| BRICS | `document.pages` | `categorias_summary` (6 categorias) | — (declaração de princípios, sem lista de ações) | — | — |
| OCDE | não informado no JSON | `categories_summary` (5 categorias) | — (recomendação de princípios) | — | — |
| AI+ (China) | `document.pages` | `key_fields` (6 campos-chave) | itens em `key_initiatives` | — | último ano em `development_targets` (2035) |
| China New Gen | não informado no JSON | `structure_overview` (6 capítulos) | `milestones` somados nas 3 fases de `strategic_objectives_timeline` | — (o documento só fixa metas de *escala de mercado* da indústria de IA, não orçamento público — por isso excluído do eixo de investimento) | último `target_year` (2030) |
| Apply AI Strategy (UE) | `document.pages` | `sectoral_flagships_summary` (11 flagships) | soma de `action_count` dos flagships | `document.funding_mobilised_eur_billion` | não informado no JSON |
| AI Continent Action Plan (UE) | `document.total_pdf_pages` | `five_key_domains` (5 domínios) | `key_actions` | "InvestAI initiative total investment mobilisation" em `key_statistics` | ano mais distante em `key_actions[].target_date` (2027) |
| America's AI Action Plan (EUA) | `document_metadata.total_pages` | `pillars` (3 pilares) | soma de `policy_actions_count_by_pillar` | não informado no JSON | não informado no JSON |

Onde o próprio documento não declara um valor, o gráfico mostra explicitamente
**"não informado no documento"** (ou a razão da exclusão, no caso do China New
Gen) — nenhum documento é omitido e nenhum valor ausente é tratado como zero.

In [ ]:
# Tabela de indicadores estruturais — construída diretamente dos campos acima,
# reaproveitando os rótulos (*_LABEL) e as cores de identidade (CATEGORICAL_*)
# já definidos nas seções anteriores para cada um dos oito documentos.

STRUCT_DOCS = [
    dict(key="brics", label=BRICS_LABEL, color=CATEGORICAL_BLUE,
         pages=7, blocks=6, blocks_unit="categorias temáticas",
         actions=0, invest_usd=None, invest_label=None,
         invest_note="declaração de princípios — não fixa orçamento",
         horizon=None, horizon_note="sem ano-alvo explícito", words=2032),
    dict(key="ocde", label=OECD_LABEL, color=CATEGORICAL_AQUA,
         pages=None, pages_note="não informado no documento",
         blocks=5, blocks_unit="categorias temáticas",
         actions=0, invest_usd=None, invest_label=None,
         invest_note="recomendação de princípios — não fixa orçamento",
         horizon=None, horizon_note="sem ano-alvo explícito", words=1938),
    dict(key="china", label=CHINA_LABEL, color=CATEGORICAL_CHINA,
         pages=None, pages_note="não informado no documento",
         blocks=6, blocks_unit="capítulos",
         actions=18, invest_usd=None, invest_label=None,
         invest_note="só fixa metas de escala de mercado, não orçamento público",
         horizon=2030, words=11077),
    dict(key="aiplus", label=AIPLUS_LABEL, color=CATEGORICAL_AIPLUS,
         pages=10, blocks=6, blocks_unit="campos-chave",
         actions=17, invest_usd=None, invest_label=None,
         invest_note="não informado no documento",
         horizon=2035, words=2670),
    dict(key="pbia", label=PBIA_LABEL, color=CATEGORICAL_PBIA,
         pages=66, blocks=5, blocks_unit="eixos estruturantes",
         actions=81, invest_usd=4.26, invest_label="R$ 23,03 bi (2024–2028)",
         horizon=2028, words=4874),
    dict(key="apply", label=APPLY_LABEL, color=CATEGORICAL_APPLY,
         pages=19, blocks=11, blocks_unit="flagships setoriais",
         actions=28, invest_usd=1.08, invest_label="€1 bi (mobilizado)",
         horizon=None, horizon_note="sem ano-alvo explícito", words=7537),
    dict(key="continent", label=CONTINENT_LABEL, color=CATEGORICAL_CONTINENT,
         pages=25, blocks=5, blocks_unit="domínios-chave",
         actions=32, invest_usd=216, invest_label="€200 bi (mobilização InvestAI)",
         horizon=2027, words=7952),
    dict(key="america", label=AMERICA_LABEL, color=CATEGORICAL_AMERICA,
         pages=25, blocks=3, blocks_unit="pilares",
         actions=103, invest_usd=None, invest_label=None,
         invest_note="não informado no documento",
         horizon=None, horizon_note="sem ano-alvo explícito", words=3636),
]
PBIA_KEY = "pbia"

# Conversão aproximada só para permitir a leitura visual conjunta no mesmo eixo:
# câmbio comercial de referência ~jul/2026 (1 USD ≈ R$ 5,40; 1 EUR ≈ US$ 1,08).
# Os valores oficiais, na moeda original, seguem nos rótulos de cada barra.

for d in STRUCT_DOCS:
    print(f"{d['label']:35s} páginas={str(d.get('pages')):>4s}  "
          f"blocos={d['blocks']:>2d} ({d['blocks_unit']})  "
          f"ações={d['actions']:>3d}  palavras={d['words']:>6d}")


### 15.1. Painel de indicadores estruturais — os oito documentos lado a lado

Seis pequenos múltiplos (mesma ordem de documentos, eixo Y compartilhado),
cada um com sua própria escala — sem eixo duplo, sem forçar unidades
diferentes na mesma régua. A ordem dos documentos não é por magnitude: começa
nas duas declarações multilaterais (BRICS, OCDE), passa pelos dois documentos
chineses, chega ao **PBIA** — destacado com uma faixa vermelha atrás da linha
inteira —, e segue para os dois documentos da UE e o dos EUA. Barras ausentes
no JSON aparecem como texto em itálico cinza ("não informado no documento"),
nunca como barra zero; zero é mostrado apenas quando é um valor real (BRICS e
OCDE realmente não listam nenhuma ação concreta — são documentos de
princípios, não de implementação).

In [ ]:
metrics = [
    ("pages", "Páginas do\ndocumento", "{:.0f}"),
    ("blocks", "Blocos estruturantes\n(eixos/pilares/domínios/capítulos)", "{:.0f}"),
    ("actions", "Ações concretas\nenumeradas", "{:.0f}"),
    ("words", "Volume textual\n(nº de palavras)", "{:.0f}"),
    ("invest_usd", "Investimento explicitado\n(US$ bi, conversão aproximada)", "{:.1f}"),
    ("horizon", "Horizonte temporal\n(ano-alvo mais distante)", "{:.0f}"),
]

fig, axes = plt.subplots(1, 6, figsize=(24, 6.4), facecolor=SURFACE)
labels_struct = [d["label"] for d in STRUCT_DOCS]
y_pos = np.arange(len(STRUCT_DOCS))[::-1]
pbia_i = [i for i, d in enumerate(STRUCT_DOCS) if d["key"] == PBIA_KEY][0]

for ax, (key, title, fmt) in zip(axes, metrics):
    ax.set_facecolor(SURFACE)
    is_horizon = key == "horizon"
    vals = [d.get(key) for d in STRUCT_DOCS]
    numeric_vals = [v for v in vals if v is not None]
    vmax = max(numeric_vals) if numeric_vals else 1

    ax.axhspan(y_pos[pbia_i] - 0.5, y_pos[pbia_i] + 0.5, color=CATEGORICAL_PBIA, alpha=0.08, zorder=0)

    if is_horizon:
        baseline = 2020
        vmax_h = max(numeric_vals) if numeric_vals else baseline + 1
        for i, d in enumerate(STRUCT_DOCS):
            y = y_pos[i]
            v = d.get(key)
            is_pbia = d["key"] == PBIA_KEY
            if v is None:
                ax.text(baseline + 0.3, y, d.get("horizon_note", "não informado"),
                        va="center", ha="left", color=INK_MUTED, fontsize=8, fontstyle="italic")
                continue
            ax.plot([baseline, v], [y, y], color=d["color"], linewidth=2.2 if is_pbia else 1.6,
                    alpha=1.0 if is_pbia else 0.7, zorder=3)
            ax.scatter([v], [y], s=70 if is_pbia else 46, color=d["color"], zorder=4,
                       edgecolors=INK_PRIMARY if is_pbia else "none", linewidths=1.1)
            ax.text(v + 0.25, y, str(v), va="center", ha="left", color=INK_PRIMARY,
                    fontsize=8.5, fontweight="bold" if is_pbia else "normal")
        ax.set_xlim(baseline, vmax_h + 3)
    else:
        for i, d in enumerate(STRUCT_DOCS):
            y = y_pos[i]
            v = d.get(key)
            is_pbia = d["key"] == PBIA_KEY
            color = d["color"]
            if v is None:
                note = d.get(f"{key}_note", "não informado no documento")
                ax.text(vmax * 0.02, y, note, va="center", ha="left",
                        color=INK_MUTED, fontsize=8, fontstyle="italic")
                continue
            if v == 0:
                ax.scatter([0], [y], marker="o", s=42, color=color, zorder=4,
                           edgecolors=INK_PRIMARY if is_pbia else "none", linewidths=1.0)
                ax.text(vmax * 0.03, y, "0", va="center", ha="left", color=INK_PRIMARY, fontsize=8.5)
                continue
            ax.barh(y, v, height=0.6, color=color, zorder=3,
                    alpha=1.0 if is_pbia else 0.85,
                    edgecolor=INK_PRIMARY if is_pbia else "none",
                    linewidth=1.1 if is_pbia else 0)
            label_txt = d.get("invest_label") if key == "invest_usd" and d.get("invest_label") else fmt.format(v)
            ax.text(v + vmax * 0.03, y, label_txt, va="center", ha="left", color=INK_PRIMARY,
                    fontsize=8.5, fontweight="bold" if is_pbia else "normal")
        ax.set_xlim(0, vmax * 1.6)

    ax.set_title(title, color=INK_PRIMARY, fontsize=10, fontweight="bold", pad=10)
    ax.set_ylim(-0.7, len(STRUCT_DOCS) - 0.3)
    ax.set_yticks(y_pos)
    if ax is axes[0]:
        ax.set_yticklabels(labels_struct, fontsize=9.5, color=INK_PRIMARY)
        for tick, d in zip(ax.get_yticklabels(), STRUCT_DOCS):
            if d["key"] == PBIA_KEY:
                tick.set_fontweight("bold")
                tick.set_color(CATEGORICAL_PBIA)
    else:
        ax.set_yticklabels([])
    ax.tick_params(axis="x", colors=INK_MUTED, labelsize=8)
    ax.tick_params(axis="y", length=0)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle(
    "Indicadores estruturais dos oito documentos de política de IA — o PBIA em destaque",
    color=INK_PRIMARY, fontsize=14.5, fontweight="bold", y=1.04,
)
fig.tight_layout()
fig.savefig("PBIA_indicadores_estruturais.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")
plt.show()


### 15.2. Mapa de posicionamento — amplitude × operacionalização, com o PBIA como referência central

Eixo X: **amplitude estrutural** (quantos blocos temáticos de alto nível o
próprio documento declara — eixo, pilar, domínio, capítulo ou campo-chave,
conforme a arquitetura que cada documento escolheu para si). Eixo Y: **grau de
operacionalização** (quantas ações, iniciativas ou marcos concretos o
documento efetivamente enumera). As linhas tracejadas vermelhas cruzam
exatamente nas coordenadas do **PBIA** — não no centro geométrico do gráfico —
e dividem o plano em quatro leituras relativas a ele: mais/menos amplo,
mais/menos operacional. É essa escolha de referência, e não a posição bruta
dos pontos, que coloca o PBIA no centro da análise.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

pbia_struct = next(d for d in STRUCT_DOCS if d["key"] == PBIA_KEY)
px, py = pbia_struct["blocks"], pbia_struct["actions"]
xmax, ymax = 13, 112

ax.axvline(px, color=CATEGORICAL_PBIA, linestyle="--", linewidth=1.3, alpha=0.5, zorder=1)
ax.axhline(py, color=CATEGORICAL_PBIA, linestyle="--", linewidth=1.3, alpha=0.5, zorder=1)

LABEL_OFFSETS = {
    "brics":     dict(xytext=(13, -16), ha="left",  va="top"),
    "ocde":      dict(xytext=(-13, -16), ha="right", va="top"),
    "china":     dict(xytext=(15, 9),   ha="left",  va="bottom"),
    "aiplus":    dict(xytext=(-15, -10), ha="right", va="top"),
    "pbia":      dict(xytext=(0, 17),   ha="center", va="bottom"),
    "apply":     dict(xytext=(0, 15),   ha="center", va="bottom"),
    "continent": dict(xytext=(-16, 10), ha="right", va="bottom"),
    "america":   dict(xytext=(15, 6),   ha="left",  va="bottom"),
}

for d in STRUCT_DOCS:
    is_pbia = d["key"] == PBIA_KEY
    x, y = d["blocks"], d["actions"]
    ax.scatter([x], [y], s=340 if is_pbia else 240, color=d["color"],
               alpha=0.95 if is_pbia else 0.82,
               edgecolors=INK_PRIMARY if is_pbia else SURFACE,
               linewidths=1.8 if is_pbia else 1.0, zorder=6 if is_pbia else 4)
    off = LABEL_OFFSETS[d["key"]]
    ax.annotate(d["label"], (x, y), textcoords="offset points", xytext=off["xytext"],
                ha=off["ha"], va=off["va"],
                fontsize=10.5 if is_pbia else 9,
                fontweight="bold" if is_pbia else "normal",
                color=CATEGORICAL_PBIA if is_pbia else INK_PRIMARY)

ax.set_xlim(0, xmax)
ax.set_ylim(-10, ymax)
ax.set_xlabel("Amplitude estrutural — nº de blocos temáticos de alto nível declarados pelo documento",
              color=INK_MUTED, fontsize=10)
ax.set_ylabel("Grau de operacionalização — nº de ações/iniciativas/marcos concretos enumerados",
              color=INK_MUTED, fontsize=10)
ax.set_title("Mapa de posicionamento — amplitude × operacionalização, com o PBIA como referência central",
             color=INK_PRIMARY, fontsize=13.5, fontweight="bold", pad=14)

ax.text(xmax * 0.99, py + ymax * 0.025, "▲ mais operacional que o PBIA",
        ha="right", va="bottom", fontsize=9, color=CATEGORICAL_PBIA, alpha=0.85)
ax.text(xmax * 0.99, py - ymax * 0.025, "▼ menos operacional que o PBIA",
        ha="right", va="top", fontsize=9, color=CATEGORICAL_PBIA, alpha=0.85)
ax.text(px + xmax * 0.012, ymax * 0.985, "mais amplo\nque o PBIA →",
        ha="left", va="top", fontsize=8.3, color=CATEGORICAL_PBIA, alpha=0.75)
ax.text(px - xmax * 0.012, ymax * 0.985, "← mais restrito\nque o PBIA",
        ha="right", va="top", fontsize=8.3, color=CATEGORICAL_PBIA, alpha=0.75)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color(GRIDLINE)
ax.spines["bottom"].set_color(GRIDLINE)
ax.tick_params(colors=INK_MUTED, labelsize=9)
ax.grid(True, color=GRIDLINE, linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig("PBIA_mapa_posicionamento.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")
plt.show()


### Síntese — o que o painel estrutural mostra que a análise de vocabulário (seções 1–14) não mostra

- **O PBIA é, dos oito documentos, o segundo mais operacionalizado**: 81 ações
  concretas enumeradas (27 de impacto imediato + 54 estruturantes), atrás
  apenas do America's AI Action Plan (103) e à frente de todos os quatro
  documentos europeus/chineses e das duas declarações multilaterais — que,
  no caso de BRICS e OCDE, não enumeram nenhuma ação (são documentos de
  princípios, não de implementação). No eixo de amplitude estrutural (5
  eixos), o PBIA fica no meio do conjunto: mais enxuto que o Apply AI
  Strategy (11 flagships) e o BRICS/AI+/China New Gen (6 blocos cada), no
  mesmo patamar da OCDE e do AI Continent Action Plan (5), e mais estruturado
  que o America's AI Action Plan (3 pilares).
- **O PBIA é o documento mais extenso em páginas** (66) entre os que
  informam esse dado — mais que o triplo do segundo colocado (AI Continent
  Action Plan e America's AI Action Plan, 25 páginas cada) — mas não o mais
  extenso em volume textual (4.874 palavras), ficando atrás de China New Gen
  (11.077), AI Continent Action Plan (7.952) e Apply AI Strategy (7.537): o
  PBIA usa mais páginas para menos palavras corridas, compatível com um
  documento rico em tabelas, listas numeradas e infográficos (structuring
  actions, breakdown de investimento) em vez de texto corrido.
- **O eixo de investimento é o menos comparável dos cinco**: apenas PBIA
  (R$ 23,03 bi), Apply AI Strategy (€1 bi) e AI Continent Action Plan
  (€200 bi) declaram uma cifra agregada, e mesmo esses três não medem a
  mesma coisa — o valor do PBIA é orçamento público comprometido para
  2024–2028, o do Apply AI Strategy é "funding mobilised" (mistura de
  fundos públicos e privados esperados), e o do AI Continent Action Plan é
  uma meta de mobilização de investimento (InvestAI) que também combina
  capital público e privado em escala continental — por isso a distância de
  quase 90× entre o valor do PBIA e o do AI Continent Action Plan deve ser
  lida como diferença de **escala geográfica e de natureza da cifra**, não
  como o Brasil investindo proporcionalmente menos que a UE.
- **Apenas três documentos fixam um ano-alvo explícito de longo prazo** além
  do horizonte do PBIA (2028): China New Gen (2030), AI Continent Action
  Plan (2027) e AI+ (2035) — BRICS, OCDE, Apply AI Strategy e America's AI
  Action Plan não declaram um horizonte temporal no texto extraído.

**Pontos fortes desta visualização:**
1. Sai da análise de vocabulário (a única lente usada nas seções 1–14) para
   uma análise de **conteúdo substantivo** — o que cada plano de fato promete
   entregar, não como ele descreve o tema — usando exclusivamente campos
   estruturados que já existem nos JSONs, sem inferência textual.
2. Nenhum documento nem nenhum valor é omitido: onde o JSON não informa um
   dado, o gráfico mostra isso explicitamente, em vez de apagar a linha ou
   preencher com zero — uma lacuna de dado nunca é visualmente confundida com
   "documento fraco nesse eixo".
3. O PBIA funciona como eixo de leitura explícito (linhas de referência no
   mapa de posicionamento, faixa destacada no painel de indicadores),
   respondendo diretamente à pergunta de pesquisa — "como o PBIA se
   posiciona frente aos outros planos" — sem recorrer a truques visuais
   (nenhum dado foi movido para forçar o PBIA ao centro geométrico do
   gráfico).

**O que fortaleceria a visualização/peso acadêmico:**
1. **Normalizar "ações concretas" por escopo textual** (ex.: ações por 1.000
   palavras) antes de comparar 81 (PBIA) com 103 (EUA) ou 18 (China New Gen)
   — hoje a contagem bruta favorece documentos que numeram explicitamente
   cada item (PBIA, EUA) frente a documentos como o China New Gen, cujos
   "marcos" são objetivos de fase, não uma lista de ações no mesmo grão. Uma
   nota metodológica ou um segundo painel com a razão ações/palavras tornaria
   a comparação mais defensável a uma banca.
2. **Converter os valores de investimento com uma fonte de câmbio datada e
   citável** (hoje a conversão USD é só para ordenar visualmente, com taxa
   aproximada) e, idealmente, **anualizar** os valores plurianuais (PBIA:
   R$23,03 bi/4 anos; AI Continent: €200 bi/~7 anos) para evitar comparar
   totais de janelas temporais diferentes.
3. **Validar a categorização "blocos estruturantes" com um segundo
   codificador** — a escolha de qual campo de cada JSON conta como "bloco de
   alto nível" (eixo vs. pilar vs. flagship vs. capítulo) é uma decisão
   interpretativa; documentá-la como matriz de codificação com critério
   explícito (já esboçado na tabela da seção 15) e, se possível, ter um
   segundo leitor validar essa codificação aumentaria a confiabilidade
   (inter-rater reliability) do indicador.
4. **Adicionar um painel de sensibilidade**: repetir o mapa de posicionamento
   (15.2) usando páginas em vez de volume textual, e ações/página em vez de
   ações brutas, para checar se o "PBIA como 2º mais operacional" se mantém
   sob diferentes definições — reforça a robustez do achado central para
   fins de publicação.

## 16. Vocabulário compartilhado entre os oito documentos — comum, específico e por subconjuntos

As seções 1–14 compararam os documentos **par a par** (BRICS×OCDE, PBIA×BRICS,
AI+×OCDE etc.). Esta seção olha para o conjunto inteiro de uma vez só e
responde a quatro perguntas sobre o vocabulário dos oito documentos:

1. **Quais termos são compartilhados por todos os oito** — o núcleo comum de
   vocabulário de política de IA, presente em BRICS, OCDE, PBIA, AI+, China
   New Gen, Apply AI Strategy, America's AI Action Plan e AI Continent Action
   Plan ao mesmo tempo;
2. **Quais termos são específicos de cada documento** — presentes em apenas
   um dos oito, e por isso os que mais o diferenciam dos demais (excluindo os
   nomes autorreferentes de país/bloco/instituição já isolados nas seções
   3, 4, 6, 8, 10 e 12, para não poluir o resultado com "brazil", "china",
   "european" etc.);
3. **Quais termos, mesmo sem ser exclusivos, são desproporcionalmente mais
   usados por um documento** — termos que também aparecem em outros
   documentos, mas de forma muito mais concentrada num deles (ex.:
   *multilateral* não é exclusivo do BRICS, mas é o BRICS quem mais o usa,
   de longe);
4. **Quais termos são compartilhados só por alguns documentos** — pares ou
   pequenos grupos (ex.: só os dois documentos da UE entre si, só os dois da
   China entre si, só Brasil e China, só Brasil e UE, só China e UE etc.).

Um termo é considerado **presente** em um documento quando ocorre pelo menos
`MIN_COUNT_PER_DOC` vezes nele — evita que uma única menção incidental conte
como "presença". A partir disso, cada termo do vocabulário combinado recebe
uma **frequência de documento** (`doc_freq`, de 1 a 8): em quantos dos oito
documentos ele de fato aparece.

In [ ]:
MIN_COUNT_PER_DOC = 2

doc_labels8 = [BRICS_LABEL, OECD_LABEL, AIPLUS_LABEL, PBIA_LABEL, CHINA_LABEL, APPLY_LABEL, AMERICA_LABEL, CONTINENT_LABEL]
doc_colors8 = [CATEGORICAL_BLUE, CATEGORICAL_AQUA, CATEGORICAL_AIPLUS, CATEGORICAL_PBIA, CATEGORICAL_CHINA, CATEGORICAL_APPLY, CATEGORICAL_AMERICA, CATEGORICAL_CONTINENT]
doc_short8 = ["BRICS", "OCDE", "AI+", "PBIA", "China\nNew Gen", "Apply AI\nStrategy", "America's AI\nAction Plan", "AI Continent\nAction Plan"]
doc_counts8 = [brics_counts, ocde_counts, aiplus_counts, pbia_counts, china_counts, apply_counts, america_counts, continent_counts]
doc_tokens8 = [brics_tokens, ocde_tokens, aiplus_tokens, pbia_tokens, china_tokens, apply_tokens, america_tokens, continent_tokens]

full_vocab8 = set()
for counts in doc_counts8:
    full_vocab8.update(counts)

term_doc_presence = {}
for t in full_vocab8:
    present = [i for i, counts in enumerate(doc_counts8) if counts.get(t, 0) >= MIN_COUNT_PER_DOC]
    if present:
        term_doc_presence[t] = present

doc_freq = {t: len(idxs) for t, idxs in term_doc_presence.items()}

doc_freq_hist = Counter(doc_freq.values())
print(f"Vocabulário combinado (>= {MIN_COUNT_PER_DOC} ocorrências em pelo menos 1 documento): {len(term_doc_presence)} termos\n")
print("Distribuição por nº de documentos em que o termo aparece:")
for df in range(8, 0, -1):
    print(f"  presente em {df} documento(s): {doc_freq_hist.get(df, 0):>4d} termos")

### 16.1. Termos comuns aos oito documentos

Termos com `doc_freq == 8` — usados por todos, do BRICS ao AI Continent
Action Plan — ranqueados pelo número total de ocorrências somado nos oito
documentos.

In [ ]:
common_terms8 = [t for t, df in doc_freq.items() if df == 8]
common_pairs8 = [
    (t, sum(counts.get(t, 0) for counts in doc_counts8))
    for t in common_terms8
]
common_pairs8.sort(key=lambda x: x[1], reverse=True)

print(f"Termos presentes (>= {MIN_COUNT_PER_DOC}x) em todos os oito documentos: {len(common_terms8)}")

plot_ranked_bar(
    common_pairs8[:30],
    "Termos comuns aos oito documentos (nº total de ocorrências)",
)

### 16.2. Termos específicos de cada documento

Termos com `doc_freq == 1` — presentes em apenas um dos oito documentos —
ranqueados pela frequência relativa (ocorrências por 1.000 tokens) dentro do
próprio documento. Nomes autorreferentes de país/bloco/instituição (já
isolados nas seções 3, 4, 6, 8, 10 e 12: *brazil*, *brazilian*, *pbia*,
*china*, *chinese*, *european*, *europe*, *eu*, *commission*, *america*,
*american*, *united*, *states*, *trump*, além de *brics*, *ocde* e *oecd*)
são excluídos aqui para evidenciar vocabulário temático, não apenas
gentílicos.

In [ ]:
SELF_REFERENTIAL_ALL = (
    SELF_REFERENTIAL | SELF_REFERENTIAL_AIPLUS | SELF_REFERENTIAL_CHINA
    | SELF_REFERENTIAL_APPLY | SELF_REFERENTIAL_CONTINENT | SELF_REFERENTIAL_AMERICA
    | {"brics", "ocde", "oecd"}
)

TOP_N_SPECIFIC = 6

specific_by_doc = [[] for _ in range(8)]
for t, idxs in term_doc_presence.items():
    if len(idxs) != 1 or t in SELF_REFERENTIAL_ALL:
        continue
    i = idxs[0]
    rel = relative_freq(doc_counts8[i], len(doc_tokens8[i]), t)
    specific_by_doc[i].append((t, rel))

for lst in specific_by_doc:
    lst.sort(key=lambda x: x[1], reverse=True)

for i, label in enumerate(doc_labels8):
    top = ", ".join(t for t, _ in specific_by_doc[i][:TOP_N_SPECIFIC])
    print(f"{label:32s} ({len(specific_by_doc[i]):>3d} termos específicos): {top}")

fig, axes = plt.subplots(2, 4, figsize=(18, 7.6), facecolor=SURFACE)
for ax, i in zip(axes.flat, range(8)):
    ax.set_facecolor(SURFACE)
    top = specific_by_doc[i][:TOP_N_SPECIFIC][::-1]
    labels = [t for t, _ in top]
    vals = [rel for _, rel in top]
    ax.barh(labels, vals, color=doc_colors8[i], height=0.6, zorder=3)
    ax.set_title(doc_labels8[i], color=doc_colors8[i], fontsize=10, fontweight="bold", pad=8)
    ax.tick_params(axis="y", labelsize=8.5, colors=INK_PRIMARY)
    ax.tick_params(axis="x", labelsize=7.5, colors=INK_MUTED)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle(
    "Termos específicos de cada documento — presentes em só um dos oito (frequência por 1.000 tokens)",
    color=INK_PRIMARY, fontsize=13.5, fontweight="bold", y=1.02,
)
fig.tight_layout()
fig.savefig("vocabulario_termos_especificos_8docs.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")
plt.show()

### 16.3. Termos mais característicos de cada documento (não exclusivos, mas dominantes)

A seção 16.2 mostrou só os termos **100% exclusivos** de cada documento
(`doc_freq == 1`). Mas um termo pode aparecer em mais de um documento e ainda
assim ser muito mais característico de um deles — o próprio *multilateral*,
por exemplo, aparece tanto no BRICS quanto em outro documento, mas quem o usa
de forma esmagadoramente mais frequente é o BRICS. Esta seção generaliza a
pergunta da 16.2 para captar esse caso.

Para cada termo com `doc_freq >= 2` (ou seja, **excluindo** os termos já
mostrados na seção 16.2), calculamos uma **taxa de dominância**: a frequência
relativa do termo no documento *i* dividida pela soma das frequências
relativas do termo nos oito documentos. Um valor de 1,0 significaria
exclusividade total (por isso já ficou de fora); um valor de 0,5 significa
que **um documento sozinho responde por metade (ou mais) de todo o uso do
termo entre os oito** — mais do que os outros sete somados. Usamos esse 0,5
como limiar de qualificação (`DOMINANCE_THRESHOLD`), exigindo também pelo
menos 2 ocorrências do termo no próprio documento (`MIN_COUNT_DOMINANT`) para
evitar que uma coincidência de baixa contagem infle a taxa.

In [ ]:
DOMINANCE_THRESHOLD = 0.5
MIN_COUNT_DOMINANT = 2
TOP_N_DOMINANT = 8

def dominance_score(t, i):
    rels = [relative_freq(doc_counts8[k], len(doc_tokens8[k]), t) for k in range(8)]
    total = sum(rels)
    return (rels[i] / total) if total > 0 else 0.0

dominant_by_doc = [[] for _ in range(8)]
for t in full_vocab8:
    if t in SELF_REFERENTIAL_ALL or doc_freq.get(t, 1) < 2:
        continue  # doc_freq < 2 -> termo exclusivo, já coberto na seção 16.2
    for i in range(8):
        c_i = doc_counts8[i].get(t, 0)
        if c_i < MIN_COUNT_DOMINANT:
            continue
        score = dominance_score(t, i)
        if score >= DOMINANCE_THRESHOLD:
            dominant_by_doc[i].append((t, score, c_i, doc_freq[t]))

for lst in dominant_by_doc:
    lst.sort(key=lambda x: (x[1], x[2]), reverse=True)

for i, label in enumerate(doc_labels8):
    top = dominant_by_doc[i][:6]
    txt = ", ".join(f"{t} ({df} docs, {score*100:.0f}%)" for t, score, c, df in top)
    print(f"{label:32s} ({len(dominant_by_doc[i]):>3d} termos qualificados): {txt}")

fig, axes = plt.subplots(2, 4, figsize=(18, 7.8), facecolor=SURFACE)
for ax, i in zip(axes.flat, range(8)):
    ax.set_facecolor(SURFACE)
    top = dominant_by_doc[i][:TOP_N_DOMINANT][::-1]
    labels = [f"{t}  ({df} docs)" for t, score, c, df in top]
    vals = [score for t, score, c, df in top]
    ax.barh(labels, vals, color=doc_colors8[i], height=0.6, zorder=3)
    ax.axvline(0.5, color=INK_MUTED, linestyle="--", linewidth=0.9, zorder=2, alpha=0.7)
    ax.set_xlim(0, 1.05)
    ax.set_title(doc_labels8[i], color=doc_colors8[i], fontsize=10, fontweight="bold", pad=8)
    ax.tick_params(axis="y", labelsize=8, colors=INK_PRIMARY)
    ax.tick_params(axis="x", labelsize=7.5, colors=INK_MUTED)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle(
    "Termos mais característicos de cada documento — não exclusivos, mas dominantes\n"
    "(taxa de dominância: share da frequência relativa do termo que pertence a este documento; rótulo = em quantos dos 8 docs o termo aparece)",
    color=INK_PRIMARY, fontsize=12, fontweight="bold", y=1.05,
)
fig.tight_layout()
fig.savefig("vocabulario_termos_dominantes_8docs.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")
plt.show()

### 16.4. Termos compartilhados só por alguns documentos

Para cada par de documentos, contamos quantos termos aparecem **exatamente
naqueles dois** (`doc_freq == 2`, com os dois índices batendo com o par) —
nem nos outros seis, nem em só um deles. É a pergunta "o que só a UE
compartilha entre si, o que só a China compartilha entre si, o que só Brasil
e China têm em comum, o que só Brasil e UE têm em comum, o que só China e UE
têm em comum" etc., respondida para todos os 28 pares possíveis de uma vez.
A diagonal reaproveita a contagem de termos específicos da seção 16.2, para
comparar lado a lado "exclusivo de 1 documento" e "exclusivo de cada par".

In [ ]:
pair_exclusive_terms = {}
for t, idxs in term_doc_presence.items():
    if len(idxs) == 2:
        i, j = idxs
        pair_exclusive_terms.setdefault((i, j), []).append(t)

for key, terms in pair_exclusive_terms.items():
    i, j = key
    terms.sort(key=lambda t: doc_counts8[i].get(t, 0) + doc_counts8[j].get(t, 0), reverse=True)

M_pairs = np.zeros((8, 8))
for i in range(8):
    M_pairs[i, i] = len(specific_by_doc[i])
for (i, j), terms in pair_exclusive_terms.items():
    M_pairs[i, j] = len(terms)
    M_pairs[j, i] = len(terms)

off_diag_max = max(M_pairs[i, j] for i in range(8) for j in range(8) if i != j)

# A diagonal (termos exclusivos de 1 só documento, até 339 no China New Gen)
# tem escala muito maior que os pares (até 74) -- se entrasse na mesma escala
# de cor, apagaria o contraste entre os pares, que é o dado que a pergunta
# pede. Por isso a diagonal fica fora da escala de cor sequencial (mascarada)
# e ganha um preenchimento neutro fixo, com o número como referência textual.
M_offdiag = np.ma.masked_array(M_pairs, mask=np.eye(8, dtype=bool))

from matplotlib.colors import LinearSegmentedColormap
SEQ_CMAP = LinearSegmentedColormap.from_list("seq_blue", [SURFACE, SEQUENTIAL_BLUE])
SEQ_CMAP.set_bad(color=GRIDLINE)

fig, ax = plt.subplots(figsize=(9.5, 8.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

im = ax.imshow(M_offdiag, cmap=SEQ_CMAP, vmin=0, vmax=off_diag_max)

ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels(doc_short8, fontsize=8.5, color=INK_PRIMARY, rotation=40, ha="right")
ax.set_yticklabels(doc_short8, fontsize=8.5, color=INK_PRIMARY)
ax.tick_params(length=0)

for i in range(8):
    for j in range(8):
        v = M_pairs[i, j]
        is_diag = i == j
        if is_diag:
            txt_color = INK_MUTED
        else:
            txt_color = INK_PRIMARY if v < off_diag_max * 0.6 else SURFACE
        ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                fontsize=9.5, fontweight="bold" if is_diag else "normal",
                color=txt_color)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2)
ax.tick_params(which="minor", length=0)

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.ax.tick_params(colors=INK_MUTED, labelsize=8)
cbar.set_label("nº de termos exclusivos do par (fora da diagonal)", color=INK_MUTED, fontsize=9)

ax.set_title(
    "Termos compartilhados só por alguns documentos — nº de termos exclusivos por par\n(diagonal, em cinza: termos exclusivos de 1 só documento — fora da escala de cor, ver seção 16.2)",
    color=INK_PRIMARY, fontsize=11.5, fontweight="bold", pad=14,
)

fig.tight_layout()
fig.savefig("vocabulario_pares_exclusivos_8docs.png", dpi=150, facecolor=SURFACE, bbox_inches="tight")
plt.show()

### 16.5. Exemplos de termos exclusivos por par

Lista concreta dos termos que sustentam algumas células do mapa acima —
grupos intra-bloco (UE com ela mesma, China com ela mesma) e os principais
cruzamentos geopolíticos (Brasil × China, Brasil × UE, Brasil × EUA,
China × UE, China × EUA, EUA × UE, BRICS × OCDE).

In [ ]:
IDX8 = {label: i for i, label in enumerate(doc_labels8)}

def pair_terms(label_a, label_b, n=6):
    i, j = IDX8[label_a], IDX8[label_b]
    key = (min(i, j), max(i, j))
    return pair_exclusive_terms.get(key, [])[:n]

highlight_pairs = [
    (APPLY_LABEL, CONTINENT_LABEL, "UE com ela mesma (Apply AI Strategy × AI Continent Action Plan)"),
    (AIPLUS_LABEL, CHINA_LABEL, "China com ela mesma (AI+ × China New Gen)"),
    (PBIA_LABEL, AIPLUS_LABEL, "Brasil × China (PBIA × AI+)"),
    (PBIA_LABEL, CHINA_LABEL, "Brasil × China (PBIA × China New Gen)"),
    (PBIA_LABEL, APPLY_LABEL, "Brasil × UE (PBIA × Apply AI Strategy)"),
    (PBIA_LABEL, CONTINENT_LABEL, "Brasil × UE (PBIA × AI Continent Action Plan)"),
    (PBIA_LABEL, AMERICA_LABEL, "Brasil × EUA (PBIA × America's AI Action Plan)"),
    (AIPLUS_LABEL, APPLY_LABEL, "China × UE (AI+ × Apply AI Strategy)"),
    (CHINA_LABEL, CONTINENT_LABEL, "China × UE (China New Gen × AI Continent Action Plan)"),
    (AMERICA_LABEL, AIPLUS_LABEL, "EUA × China (America's AI Action Plan × AI+)"),
    (AMERICA_LABEL, CHINA_LABEL, "EUA × China (America's AI Action Plan × China New Gen)"),
    (AMERICA_LABEL, APPLY_LABEL, "EUA × UE (America's AI Action Plan × Apply AI Strategy)"),
    (AMERICA_LABEL, CONTINENT_LABEL, "EUA × UE (America's AI Action Plan × AI Continent Action Plan)"),
    (BRICS_LABEL, OECD_LABEL, "BRICS × OCDE"),
]

print("Termos exclusivos por par (top 6, ordenados por ocorrência combinada):\n")
for a, b, desc in highlight_pairs:
    terms = pair_terms(a, b)
    print(f"{desc}:")
    print(f"  {', '.join(terms) if terms else '(nenhum termo exclusivo acima do limiar)'}\n")

### Síntese — o que o vocabulário comum, específico e por subconjuntos revela

- **Núcleo comum enxuto: 21 termos em 1.934** (1,1% do vocabulário combinado)
  aparecem nos oito documentos ao mesmo tempo — *development* (329
  ocorrências somadas), *data* (249), *research* (193), *support* (169),
  *innovation* (159), *public*, *promote*, *national*, *intelligence*,
  *security*, *use*, *applications*, *develop*, *international*, *global*,
  *governance*, *information*, *capabilities*, *skills*, *work* e
  *resources*. É um vocabulário genérico de política pública de tecnologia —
  nenhum termo tecnicamente específico de IA (como *model*, *algorithm* ou
  *training*) é comum aos oito, sinal de que a convergência entre os
  documentos está no nível do enquadramento político (desenvolvimento,
  inovação, segurança, governança), não no vocabulário técnico.
- **A maioria do vocabulário é específica de um único documento**: 1.034 dos
  1.934 termos (53%) aparecem em só um dos oito, contra apenas 21 (1,1%) nos
  oito. A distribuição cai de forma monotônica e acentuada conforme mais
  documentos precisam compartilhar o termo (393 em exatamente 2 documentos,
  213 em 3, 106 em 4, 73 em 5, 60 em 6, 34 em 7) — o vocabulário de política
  de IA, no conjunto, é mais fragmentado por documento do que compartilhado.
- **O tamanho do documento pesa na contagem de termos específicos**: o China
  New Gen, de longe o mais extenso (11.077 palavras — ver seção 15), também
  tem de longe mais termos exclusivos (339, quase o triplo do segundo
  colocado) e as maiores contagens de termos exclusivos em quase todos os
  seus pares (32 com o PBIA, 44 com o Apply AI Strategy, 31 com o AI
  Continent Action Plan, 23 com o America's AI Action Plan). Isso é
  esperado estatisticamente — mais texto gera mais vocabulário raro — e deve
  ser lido como um efeito de volume textual, não como evidência de que a
  China "compartilha mais" tematicamente; os documentos mais curtos (BRICS,
  2.032 palavras; OECD, 1.938) têm poucos termos específicos (25 e 31) só
  porque têm pouco vocabulário no total.
- **O par mais coeso do conjunto inteiro é o par institucional europeu**
  (Apply AI Strategy × AI Continent Action Plan, 74 termos exclusivos —
  *commission*, *europe*, *member*, *uptake*, *facilitate*, *gigafactories*),
  seguido do par chinês (AI+ × China New Gen, 51 termos exclusivos —
  *intelligent*, *basic*, *methods*, *unmanned*, *intelligentization*).
  Confirma, agora pelo lado do vocabulário exclusivo, o mesmo agrupamento já
  visto pela similaridade de cosseno nas seções 9 e 13: os dois documentos de
  cada bloco (UE, China) formam pares muito mais próximos entre si do que
  com qualquer documento de fora do bloco.
- **Brasil (PBIA) se conecta mais com a China do que com a UE ou os EUA em
  termos exclusivos**: PBIA × China New Gen tem 32 termos exclusivos
  (*country*, *understanding*, *problems*, *nation*, *term*, *active*),
  acima de PBIA × Apply AI Strategy (19: *various*, *however*, *limited*,
  *remains*, *transport*), PBIA × AI Continent Action Plan (17:
  *investments*, *population*, *computational*, *supercomputers*, *five*,
  *vision*) e PBIA × America's AI Action Plan (9: *federal*, *agencies*,
  *requiring*, *clean*, *revolution*). O padrão é consistente com a seção
  4.3–13 (PBIA mais próximo do eixo BRICS/desenvolvimento do que do eixo
  OCDE/governança), mas aqui aparece em vocabulário concreto, não só em
  índice de similaridade.
- **A OCDE é o documento mais isolado do conjunto também nesta lente**: o
  par OECD × AI+ tem **zero** termos exclusivos — nenhuma palavra do
  vocabulário combinado aparece só nesses dois documentos — e os pares
  OECD × America's AI Action Plan (1), BRICS × AI+ (1), OECD × PBIA (3) e
  BRICS × OECD (3) estão entre os mais baixos de todo o mapa. Reforça, pelo
  ângulo do vocabulário exclusivo, o achado já visto nas seções 1 e 13: a
  OCDE fala uma língua de governança de sistemas que quase não se cruza,
  termo a termo, com a linguagem de desenvolvimento tecnológico nacional dos
  demais sete documentos.
- **Os termos específicos, isolados de nomes autorreferentes de
  país/bloco/instituição, mostram a ênfase real de cada documento**: BRICS
  pende para princípios de equidade (*equitable*, *meaningful*, *safeguard*,
  *emissions*); a OCDE, para a arquitetura formal da própria recomendação
  (*governments*, *recommendation*, *recognising*, *stewardship*); o AI+
  chinês, para verbos de execução técnica (*refine*, *exploration*,
  *execution*, *ai-native*); o PBIA, para a arquitetura do próprio plano
  (*structuring*, *sustainability*, *represents*); o China New Gen, para
  ciência de base (*theory*, *sensing*, *brain-inspired*); o Apply AI
  Strategy, para aplicação setorial e defesa (*defence*, *ai-based*,
  *clinical*, *general-purpose*); o America's AI Action Plan, para controle
  estratégico (*export*, *semiconductor*, *evaluations*, *adversaries*); e o
  AI Continent Action Plan, para infraestrutura física europeia (*eurohpc*,
  *centre*, *academy*, *spaces*).

- **Ampliando a lente para termos não exclusivos, mas dominantes (seção
  16.3), aparece um segundo nível de identidade temática**: o BRICS é o
  documento que mais concentra vocabulário de ação coletiva/multilateral
  (*accountability*, *accordance*, *fragmentation*, *domestic*, todos com
  70-80% da frequência relativa combinada dos oito documentos, mesmo
  aparecendo em mais de um deles); a OCDE concentra vocabulário de ciclo de
  vida e confiança do próprio sistema de IA (*lifecycle*, *trustworthy*,
  *consistent*); o AI+ chinês concentra vocabulário de interação
  humano-máquina (*human-computer*, *intelligentized*, *terminal*); e o
  PBIA concentra vocabulário de escala e prazo (*population*, *country*,
  *short*, *term*). Ou seja: mesmo quando um termo é usado por dois ou três
  documentos, ele ainda tende a ter um "dono" claro em termos de volume de
  uso -- a distintividade do vocabulário de cada documento não depende de
  esse vocabulário ser exclusivo, só de ser desproporcional.

**O que fortaleceria esta análise para fins acadêmicos:**
1. Normalizar a contagem de termos exclusivos por par pelo tamanho combinado
   dos dois documentos (termos exclusivos por 1.000 tokens do par, não
   contagem bruta) — hoje o China New Gen domina o mapa em parte porque é o
   documento mais longo, não necessariamente porque compartilha
   proporcionalmente mais vocabulário.
2. Testar a robustez de `MIN_COUNT_PER_DOC = 2` (o limiar que define
   "presença" de um termo num documento): repetir a seção com o limiar em 1
   e em 3 e verificar se o núcleo comum de 21 termos e os pares mais/menos
   coesos se mantêm — o resultado hoje depende dessa escolha metodológica,
   ainda que ela siga o mesmo padrão de limiar mínimo já usado nas seções
   1–14 (`MIN_COMBINED_COUNT`).
3. Complementar a contagem bruta de termos exclusivos com uma métrica
   ponderada por frequência relativa (como o `relative_freq` já usado nas
   seções 2, 4.2, 6.2 etc.), para diferenciar um par que compartilha termos
   *muito citados nos dois* de um par que compartilha termos *citados uma
   vez em cada*.

## 17. PBIA entre três blocos geopolíticos — China, União Europeia e Estados Unidos

Nas seções 1–16 cada documento internacional entrou isoladamente na comparação
(BRICS, OCDE, AI+, China New Gen, Apply AI Strategy, AI Continent Action Plan,
America's AI Action Plan). Esta seção final agrupa esses sete documentos em
**três blocos geopolíticos**, cada um tratado como um único corpus combinado —
exatamente como já fizemos para o mapa de quadrantes da Seção 14:

- **China** — `China_New_Gen.json` + `AI_PLUS.json` (plano-mestre de 2017 +
  aprofundamento setorial de 2023);
- **União Europeia** — `AI_CONTINENT_ACTION_PLAN.json` + `Apply_AI_Strategy.json`
  (as duas comunicações da Comissão Europeia de 2025);
- **Estados Unidos** — `AMERICA_AI_ACTION_PLAN.json` (já é um único documento).

Reaproveitamos os corpora combinados `china_group_counts`/`china_group_total` e
`eu_group_counts`/`eu_group_total` já construídos na Seção 14, e
`america_counts`/`america_tokens` das seções anteriores — nenhum documento é
recarregado. A pergunta desta seção: em termos **relativos**, e não absolutos
(os três blocos e o PBIA têm volumes de texto bem diferentes entre si, por
isso toda a comparação usa frequência relativa por 1.000 tokens, nunca
contagem bruta — a mesma base já usada em todo o notebook), o PBIA se
aproxima mais de um desses três blocos, ou ocupa uma posição mais autônoma,
equidistante dos três?

In [ ]:
def cosine_sim_counts(counts_a, total_a, counts_b, total_b):
    """Mesmo cálculo de cosseno de jaccard_cosine (Seção 1), mas recebendo
    contagem e total de tokens diretamente — necessário aqui porque
    china_group_counts/eu_group_counts (Seção 14) são corpora combinados de
    dois documentos, sem uma lista de tokens única para medir len()."""
    vocab = set(counts_a) | set(counts_b)
    va = [relative_freq(counts_a, total_a, t) for t in vocab]
    vb = [relative_freq(counts_b, total_b, t) for t in vocab]
    dot = sum(x * y for x, y in zip(va, vb))
    norm_a = math.sqrt(sum(x * x for x in va))
    norm_b = math.sqrt(sum(y * y for y in vb))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

BLOCK_CHINA_LABEL = "China (China New Gen + AI+)"
BLOCK_EU_LABEL = "União Europeia (AI Continent Action Plan + Apply AI Strategy)"
BLOCK_USA_LABEL = "Estados Unidos (America's AI Action Plan)"

cosine_pbia_block_china = cosine_sim_counts(pbia_counts, len(pbia_tokens), china_group_counts, china_group_total)
cosine_pbia_block_eu = cosine_sim_counts(pbia_counts, len(pbia_tokens), eu_group_counts, eu_group_total)
cosine_pbia_block_usa = cosine_sim_counts(pbia_counts, len(pbia_tokens), america_counts, len(america_tokens))

print(f"PBIA × {BLOCK_CHINA_LABEL}: cosseno = {cosine_pbia_block_china:.3f}")
print(f"PBIA × {BLOCK_EU_LABEL}: cosseno = {cosine_pbia_block_eu:.3f}")
print(f"PBIA × {BLOCK_USA_LABEL}: cosseno = {cosine_pbia_block_usa:.3f}")


### Gráfico — com qual bloco o PBIA tem maior similaridade de ênfase textual?

As três similaridades de cosseno lado a lado, com as mesmas cores de
identidade já usadas para UE, China e EUA no mapa de quadrantes da Seção 14.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

block_labels = [BLOCK_CHINA_LABEL, BLOCK_EU_LABEL, BLOCK_USA_LABEL]
block_values = [cosine_pbia_block_china, cosine_pbia_block_eu, cosine_pbia_block_usa]
block_colors = [CATEGORICAL_CHINA_Q, CATEGORICAL_UE_Q, CATEGORICAL_AMERICA_Q]

y_pos = list(range(len(block_labels)))
ax.barh(y_pos, block_values, color=block_colors, height=0.55, zorder=3)
for y, v in zip(y_pos, block_values):
    ax.text(v + 0.01, y, f"{v:.3f}", va="center", ha="left", color=INK_PRIMARY, fontsize=10, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(block_labels, color=INK_PRIMARY, fontsize=10)
ax.set_xlabel("Similaridade de cosseno do PBIA com o bloco (0 a 1)", color=INK_MUTED, fontsize=10)
ax.set_xlim(0, max(block_values) * 1.35)
ax.set_title("Com qual bloco o PBIA tem maior similaridade de ênfase textual?", color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=14)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


### 17.2. Mapa ternário — o posicionamento relativo do PBIA entre os três blocos

As três similaridades de cosseno acima são normalizadas para somar 100%
(`peso do bloco = similaridade do bloco / soma das três similaridades`) e
projetadas como coordenadas baricêntricas num **diagrama ternário** — a
técnica padrão para visualizar uma composição de três partes que soma um
todo, aqui usada para representar de forma explicitamente **relativa** (não
absoluta) a ênfase textual do PBIA. Cada vértice do triângulo representa
100% de peso num único bloco; o centro do triângulo é o ponto de
**equidistância perfeita** (33,3% / 33,3% / 33,3%) — a posição mais autônoma
possível dentro deste espaço de três blocos. Quanto mais perto de um vértice
o PBIA aparecer, mais sua ênfase textual pende para aquele bloco; quanto
mais perto do centro, mais autônoma (não alinhada a nenhum bloco específico
em detrimento dos outros) é sua posição. Reportamos também um **índice de
autonomia** — 1 menos a distância do PBIA até o centro, normalizada pela
distância de qualquer vértice até o centro — que vai de 0 (colado a um único
bloco) a 1 (exatamente equidistante dos três).

In [ ]:
TRI_CHINA = np.array([1.0, 0.0])
TRI_EU = np.array([0.0, 0.0])
TRI_USA = np.array([0.5, np.sqrt(3) / 2])

def bary_to_cart(w_china, w_eu, w_usa):
    return w_china * TRI_CHINA + w_eu * TRI_EU + w_usa * TRI_USA

weights_sum = cosine_pbia_block_china + cosine_pbia_block_eu + cosine_pbia_block_usa
w_china = cosine_pbia_block_china / weights_sum
w_eu = cosine_pbia_block_eu / weights_sum
w_usa = cosine_pbia_block_usa / weights_sum

pbia_point = bary_to_cart(w_china, w_eu, w_usa)
centroid = bary_to_cart(1 / 3, 1 / 3, 1 / 3)
r_max = np.linalg.norm(TRI_CHINA - centroid)  # distância do centro a qualquer vértice (triângulo equilátero)
autonomy_index = 1 - np.linalg.norm(pbia_point - centroid) / r_max

print(f"Pesos relativos do PBIA — China: {w_china:.3f}  UE: {w_eu:.3f}  EUA: {w_usa:.3f}")
print(f"Índice de autonomia (0 = colado a um bloco, 1 = equidistante dos três): {autonomy_index:.3f}")

fig, ax = plt.subplots(figsize=(8.5, 8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.set_aspect("equal")

triangle_pts = np.array([TRI_CHINA, TRI_EU, TRI_USA, TRI_CHINA])
ax.plot(triangle_pts[:, 0], triangle_pts[:, 1], color=INK_MUTED, linewidth=1.4, zorder=2)

# Malha ternária (25/50/75%) — linhas paralelas a cada lado do triângulo
for level in (0.25, 0.5, 0.75):
    p0, p1 = bary_to_cart(level, 1 - level, 0), bary_to_cart(level, 0, 1 - level)
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=GRIDLINE, linewidth=0.8, zorder=1)
    p0, p1 = bary_to_cart(1 - level, level, 0), bary_to_cart(0, level, 1 - level)
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=GRIDLINE, linewidth=0.8, zorder=1)
    p0, p1 = bary_to_cart(1 - level, 0, level), bary_to_cart(0, 1 - level, level)
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=GRIDLINE, linewidth=0.8, zorder=1)

for point, label, color, cos_val in [
    (TRI_CHINA, BLOCK_CHINA_LABEL, CATEGORICAL_CHINA_Q, cosine_pbia_block_china),
    (TRI_EU, BLOCK_EU_LABEL, CATEGORICAL_UE_Q, cosine_pbia_block_eu),
    (TRI_USA, BLOCK_USA_LABEL, CATEGORICAL_AMERICA_Q, cosine_pbia_block_usa),
]:
    ax.scatter(*point, s=180, color=color, zorder=4, edgecolors=SURFACE, linewidths=1.2)
    offset = (0, -26) if point[1] < 0.1 else (0, 16)
    va = "top" if offset[1] < 0 else "bottom"
    ax.annotate(f"{label}\n(cosseno PBIA = {cos_val:.3f})", point, textcoords="offset points",
                xytext=offset, ha="center", va=va, fontsize=9.5, fontweight="bold", color=color)

ax.scatter(*centroid, s=90, color=INK_MUTED, marker="x", zorder=4, linewidths=2,
           label="Posição neutra (33,3% / 33,3% / 33,3%) — equidistante dos três blocos")

ax.plot([centroid[0], pbia_point[0]], [centroid[1], pbia_point[1]],
        linestyle=":", linewidth=1.3, color=CATEGORICAL_PBIA, zorder=3)

ax.scatter(*pbia_point, s=420, color=CATEGORICAL_PBIA, alpha=0.92, edgecolors=INK_PRIMARY,
           linewidths=1.3, zorder=5, marker="D")
ax.annotate(
    f"PBIA — China {w_china * 100:.0f}% · UE {w_eu * 100:.0f}% · EUA {w_usa * 100:.0f}%",
    pbia_point, textcoords="offset points", xytext=(0, 34), ha="center",
    fontsize=10.5, fontweight="bold", color=INK_PRIMARY, zorder=6,
)

ax.legend(loc="upper right", frameon=False, labelcolor=INK_MUTED, fontsize=8.5, handletextpad=0.5)

ax.set_title(
    "Posicionamento relativo do PBIA entre os três blocos — China, União Europeia e Estados Unidos",
    color=INK_PRIMARY, fontsize=13, fontweight="bold", pad=18,
)
ax.set_xlim(-0.18, 1.18)
ax.set_ylim(-0.22, TRI_USA[1] + 0.18)
ax.axis("off")

fig.text(0.5, 0.02,
         f"Índice de autonomia (distância ao centro): {autonomy_index:.2f}  —  "
         "1,00 = equidistante dos três blocos · 0,00 = colado a um único bloco",
         ha="center", fontsize=9, color=INK_MUTED)

fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


### Síntese — o PBIA entre China, União Europeia e Estados Unidos

- Nas três comparações, as similaridades de cosseno ficam muito próximas
  entre si: **0,511 com a UE**, **0,481 com a China** e **0,457 com os
  EUA** — uma amplitude de apenas 0,054 entre o bloco mais próximo e o mais
  distante, a menor de qualquer comparação de três (ou mais) documentos
  feita neste notebook. O PBIA não "gruda" fortemente a nenhum dos três
  blocos.
- Normalizando essas três similaridades em pesos relativos, o PBIA fica com
  **33,2% de peso no eixo China, 35,3% na UE e 31,5% nos EUA** — a poucos
  pontos percentuais da divisão perfeitamente equilibrada de 33,3/33,3/33,3.
  No mapa ternário, o ponto do PBIA aparece praticamente sobre o centro do
  triângulo, com um **índice de autonomia de 0,967** (numa escala de 0,
  colado a um único bloco, a 1, equidistante dos três) — o maior grau de
  "não alinhamento" observado em qualquer uma das visualizações deste
  notebook.
- Essa leitura relativa é confirmada no nível do vocabulário: entre os 25
  termos mais usados pelo PBIA (excluindo autorreferências), **10 pendem
  para o eixo China** (*development*, *intelligence*, *technological*,
  *new*, *innovation*, *technology*, entre outros — vocabulário de
  desenvolvimento tecnológico), **10 para o eixo UE** (*public*, *data*,
  *actions*, *sectors*, *responsible*, *use*, entre outros — vocabulário
  setorial e de governança) e apenas **5 para o eixo EUA** (*artificial*,
  *national*, *plan*, *impact*, *global* — vocabulário de plano executivo e
  liderança nacional). A divisão quase 10/10/5 no vocabulário espelha o
  equilíbrio já visto nas similaridades agregadas por bloco.
- Em conjunto, os resultados desta seção **qualificam, sem contradizer**, a
  conclusão das Seções 5/7/9/11 (de que o PBIA se aproxima mais do BRICS do
  que da OCDE): aquele resultado media o PBIA contra dois documentos de
  *arquitetura de governança* (BRICS e OCDE). Quando a comparação passa a
  ser feita diretamente contra os três blocos geopolíticos de IA mais
  influentes de 2025 — China, UE e EUA, sem o BRICS como referência —, o
  PBIA não reproduz de forma dominante o vocabulário de nenhum dos três. Ele
  ocupa, nesse espaço de três blocos, uma posição textualmente **autônoma**:
  levemente mais próxima da moldura setorial e regulatória europeia do que
  da retórica de desenvolvimento tecnológico chinesa ou do enquadramento de
  liderança nacional americano — mas por uma margem pequena demais para
  caracterizar um alinhamento de bloco.

### 17.3. Quais termos puxam o PBIA para cada bloco — e quais o mantêm autônomo?

O índice de autonomia da Seção 17.2 é agregado (uma similaridade de cosseno
por bloco). Esta subseção decompõe esse agregado termo a termo: para cada
palavra que o PBIA usa com frequência (mínimo de 3 ocorrências, excluindo
autorreferências — reaproveitando `SELF_REFERENTIAL_ALL` da Seção 16),
comparamos sua frequência relativa nos três blocos (China, UE, EUA) e
identificamos qual bloco usa mais aquele termo, proporcionalmente ao próprio
tamanho.

- Se a frequência relativa do segundo colocado é **inferior a 75%** da do
  primeiro, o termo "puxa" claramente para aquele bloco — é classificado no
  eixo correspondente.
- Se a proporção é **igual ou superior a 75%** (o mesmo limiar de
  *convergência* usado no mapa de quadrantes da Seção 14), os três blocos
  usam o termo de forma parecida — ele não empurra o PBIA para nenhum
  vértice do triângulo, e por isso é classificado como **autônomo**: parte
  do vocabulário comum à agenda internacional de IA como um todo, não a um
  bloco específico.

O gráfico abaixo mostra, para cada uma das quatro categorias, os termos mais
usados pelo PBIA que se encaixam nela — ranqueados pela própria frequência
do termo no PBIA, para destacar o vocabulário que mais pesa na posição do
documento.

In [ ]:
BALANCE_RATIO_TERMS = 0.75   # mesmo limiar de "convergência" da Seção 14
MIN_PBIA_COUNT_TERMS = 3
TOP_N_PER_GROUP = 10

pull_rows = []
for t, c in pbia_counts.most_common():
    if t in SELF_REFERENTIAL_ALL or c < MIN_PBIA_COUNT_TERMS:
        continue
    r_china = relative_freq(china_group_counts, china_group_total, t)
    r_eu = relative_freq(eu_group_counts, eu_group_total, t)
    r_usa = relative_freq(america_counts, len(america_tokens), t)
    vals = {"China": r_china, "UE": r_eu, "EUA": r_usa}
    if sum(vals.values()) == 0:
        continue  # termo do PBIA sem qualquer presença nos três blocos de referência
    ranked = sorted(vals.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_val = ranked[0]
    ratio = ranked[1][1] / top_val if top_val > 0 else 0.0
    group = "Autônomo" if ratio >= BALANCE_RATIO_TERMS else top_label
    pull_rows.append((t, c, r_china, r_eu, r_usa, group, ratio))

groups_pull = {"China": [], "UE": [], "EUA": [], "Autônomo": []}
for row in pull_rows:
    groups_pull[row[5]].append(row)
for g in groups_pull:
    groups_pull[g].sort(key=lambda r: r[1], reverse=True)  # ranqueado pela frequência no PBIA

print(f"Termos candidatos (freq. PBIA ≥ {MIN_PBIA_COUNT_TERMS}, presentes em ao menos 1 bloco): {len(pull_rows)}")
for g in ["China", "UE", "EUA", "Autônomo"]:
    print(f"  {g:10s}: {len(groups_pull[g])} termos")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10.5), facecolor=SURFACE)
fig.suptitle("Quais termos do PBIA puxam para cada bloco — e quais o mantêm autônomo?",
             color=INK_PRIMARY, fontsize=14, fontweight="bold", y=0.985)

panel_specs = [
    (axes[0, 0], "China", CATEGORICAL_CHINA_Q, 2, BLOCK_CHINA_LABEL),
    (axes[0, 1], "UE", CATEGORICAL_UE_Q, 3, BLOCK_EU_LABEL),
    (axes[1, 0], "EUA", CATEGORICAL_AMERICA_Q, 4, BLOCK_USA_LABEL),
]

for ax, group, color, val_idx, block_label in panel_specs:
    ax.set_facecolor(SURFACE)
    rows = groups_pull[group][:TOP_N_PER_GROUP][::-1]  # maior no topo
    labels = [r[0] for r in rows]
    values = [r[val_idx] for r in rows]
    pbia_n = [r[1] for r in rows]
    y_pos = range(len(rows))
    ax.barh(y_pos, values, color=color, height=0.62, zorder=3)
    for y, v, pc in zip(y_pos, values, pbia_n):
        ax.text(v + max(values) * 0.03, y, f"{v:.1f}  (PBIA: {pc}×)", va="center", ha="left",
                color=INK_PRIMARY, fontsize=8.5)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9.5)
    ax.set_xlim(0, max(values) * 1.4)
    ax.set_title(f"Puxam para {block_label}", color=color, fontsize=11, fontweight="bold", pad=10)
    ax.set_xlabel("Frequência relativa no bloco (por 1.000 tokens)", color=INK_MUTED, fontsize=8.5)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)

# Painel autônomo — três barras por termo (China / UE / EUA), para mostrar visualmente o equilíbrio
ax = axes[1, 1]
ax.set_facecolor(SURFACE)
auto_rows = groups_pull["Autônomo"][:8][::-1]
labels = [r[0] for r in auto_rows]
y_pos = np.arange(len(auto_rows))
bar_h = 0.24
ax.barh(y_pos + bar_h, [r[2] for r in auto_rows], height=bar_h, color=CATEGORICAL_CHINA_Q, zorder=3, label="China")
ax.barh(y_pos, [r[3] for r in auto_rows], height=bar_h, color=CATEGORICAL_UE_Q, zorder=3, label="UE")
ax.barh(y_pos - bar_h, [r[4] for r in auto_rows], height=bar_h, color=CATEGORICAL_AMERICA_Q, zorder=3, label="EUA")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, color=INK_PRIMARY, fontsize=9.5)
max_auto = max(max(r[2], r[3], r[4]) for r in auto_rows)
ax.set_xlim(0, max_auto * 1.35)
ax.set_title("Mantêm o PBIA autônomo (equilibrados entre os 3 blocos)", color=INK_MUTED, fontsize=11, fontweight="bold", pad=10)
ax.set_xlabel("Frequência relativa em cada bloco (por 1.000 tokens)", color=INK_MUTED, fontsize=8.5)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(GRIDLINE)
ax.xaxis.grid(True, color=GRIDLINE, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, labelcolor=INK_PRIMARY, fontsize=8.5)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### Síntese — o vocabulário por trás do índice de autonomia

- **Puxam para a China**: *development* (39×), *intelligence* (29×),
  *technological* (29×), *new* (19×), *innovation* (16×), *country* (16×),
  *technology* (11×) — o núcleo é vocabulário de **desenvolvimento
  tecnológico e inovação nacional**, com destaque para *development*, cuja
  frequência relativa no bloco China (17,8‰) é mais do que o dobro da UE
  (7,9‰) e quase o triplo dos EUA (6,1‰): é o termo mais "puramente chinês"
  entre os mais usados pelo PBIA.
- **Puxam para a UE**: *public* (23×), *challenges* (21×), *actions* (20×),
  *potential* (17×), *sectors* (13×), *sovereignty* (11×), *solutions*
  (10×) — vocabulário de **implementação setorial e política pública**. O
  caso mais revelador é *sovereignty*: usado 11 vezes pelo PBIA, aparece no
  bloco UE (1,4‰) mas tem frequência **zero** tanto no bloco China quanto
  nos EUA — a moldura de "soberania tecnológica/digital", central ao
  discurso europeu recente, é adotada pelo PBIA sem equivalente nos outros
  dois blocos.
- **Puxam para os EUA**: *artificial* (29×), *national* (20×), *plan*
  (18×), *global* (12×), *infrastructure* (11×), *security* (8×) —
  vocabulário de **plano executivo nacional e infraestrutura**, ecoando o
  enquadramento de "corrida" do America's AI Action Plan. Vale notar que é
  o eixo com **menos termos entre os mais frequentes do PBIA**: dos 10
  termos mais usados pelo PBIA no geral, apenas 2 (*artificial*, *national*)
  puxam para os EUA, contra 4 para a China (*development*, *intelligence*,
  *technological*, *new*) e 3 para a UE (*public*, *challenges*, *actions*)
  — consistente com os EUA sendo o bloco de menor similaridade agregada na
  Seção 17.1.
- **Mantêm o PBIA autônomo**: *data* (20×), *technologies* (16×), *impact*
  (13×), *responsible* (12×), *use* (12×), *significant* (12×),
  *sustainable* (11×), *strategic* (10×) — este é o vocabulário-base da
  agenda internacional de IA como um todo, usado de forma parecida pelos
  três blocos (a razão entre o segundo e o primeiro colocado passa de 0,75
  em todos esses termos, chegando a 0,97 em *impact* e 0,94 em
  *responsible*/*significant* — quase idêntico nos três blocos). É
  significativo que termos ligados a **governança responsável** (*responsible*,
  *governance*, *significant*) caiam nesta categoria "neutra": não é que o
  PBIA seja indiferente a governança — é que, ao contrário do padrão
  BRICS/OCDE das seções 1–13, nenhum dos três blocos geopolíticos (China,
  UE, EUA) monopoliza esse vocabulário nesta comparação.
- **Para o argumento acadêmico**: o índice de autonomia de 0,97 (Seção 17.2)
  não é um artefato de o PBIA usar pouco vocabulário "alinhável" — pelo
  contrário, há dezenas de termos puxando em cada uma das três direções
  (61 termos para o eixo China, 84 para o eixo UE, 49 para o eixo EUA, entre
  os 262 termos do PBIA analisados). A autonomia emerge porque essas
  puxadas são **numerosas e comparáveis em intensidade nos três eixos ao
  mesmo tempo**, e não porque o documento evita se posicionar. O PBIA
  absorve simultaneamente o vocabulário desenvolvimentista chinês, o
  vocabulário setorial/regulatório europeu (inclusive a retórica de
  soberania) e o vocabulário de plano nacional americano, sem que nenhum
  predomine — um retrato lexical de uma política que dialoga com os três
  blocos ao mesmo tempo, em vez de replicar o discurso de um único deles.